# Types & Shadows — Libby's Voice Narrator
### 100 Days in the Word with Libby Mays

This notebook uses **your existing Libby voice recording** as the reference voice and generates narration for all 100 devotionals.

It is built to be **resume-safe**:
- Each finished day is saved to the Colab session immediately.
- If Colab disconnects, run the notebook again; already-finished days are skipped.
- It creates both a WAV master and an MP3 for the website.
- It also saves the exact narration script used for every day.

### Before you run it
In Google Colab choose **Runtime → Change runtime type → T4 GPU**.

Then choose **Runtime → Run all**.

No Google Drive connection is required.


In [ ]:
# =========================
# LIBBY VOICE SETTINGS
# =========================

START_DAY = 1
END_DAY = 100

# These are the same voice-clone settings used in the earlier Libby preview.
EXAGGERATION = 0.55
CFG_WEIGHT = 0.30
TEMPERATURE = 0.80

# Small pauses make the devotional sound intentional instead of rushed.
PAUSE_BETWEEN_CHUNKS_MS = 325
PAUSE_BETWEEN_SECTIONS_MS = 650

# Reference voice
VOICE_URL = "https://types-and-shadows-100-days.netlify.app/libby-intro.m4a"
REFERENCE_SECONDS = 14

# Output folder in your Google Drive
OUTPUT_FOLDER = "/content/drive/MyDrive/Types-and-Shadows-Audio"

print("Settings ready.")
print(f"Days selected: {START_DAY} through {END_DAY}")


In [ ]:
import os, sys, subprocess

print("1/5 — Checking GPU...")
try:
    subprocess.run(["nvidia-smi"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception:
    raise RuntimeError("No GPU is active. In Colab choose Runtime → Change runtime type → T4 GPU → Save, then run again.")

print("GPU ready.")
print("2/5 — Installing Chatterbox voice cloning software...")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chatterbox-tts==0.1.7"], check=True)

print("Voice software installed.")


In [ ]:
import os, urllib.request, subprocess
from IPython.display import Audio, display

print("3/5 — Connecting Google Drive...")
drive.mount("/content/drive", force_remount=False)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Getting Libby's real voice sample...")
urllib.request.urlretrieve(VOICE_URL, "/content/libby-intro.m4a")

subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-ss", "0", "-t", str(REFERENCE_SECONDS),
    "-i", "/content/libby-intro.m4a",
    "-ac", "1", "-ar", "24000",
    "/content/libby-reference.wav"
], check=True)

print("Libby reference clip ready. Press play if you want to confirm the source voice:")
display(Audio("/content/libby-reference.wav"))


In [ ]:
print("4/5 — Loading the Chatterbox voice model...")
print("The first load can take several minutes.")

import torch
import torchaudio
from chatterbox.tts import ChatterboxTTS

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not active. Re-select Runtime → Change runtime type → T4 GPU.")

model = ChatterboxTTS.from_pretrained("cuda")
print("Voice model loaded and ready.")


In [ ]:
NARRATION_DATA = [{"day": 1, "title": "God Is Not Finished With You", "reference": "Luke 15:11–32; Psalm 51:10–17", "focus": "The prodigal son’s turning point came when he stopped defending where he was and came home. Biblical repentance does not erase the seriousness of the past; it refuses to let the past become the master of the future. The Father’s welcome shows the heart of grace: God receives the repentant, restores relationship, and begins forming a different person.", "pattern": "The younger son is a picture of the human heart wandering from the Father. His homecoming echoes the gospel pattern: lost, awakened, repentant, received.", "hidden": "Luke 15 says the son rehearsed a servant speech, but the father interrupts the shame-script with robe, ring, sandals, and feast. Grace restores relationship before the son can earn his way back.", "why": "God receives the repentant and rebuilds from the inside out.", "reflect": "Where have I been explaining myself when God is asking me simply to come home?", "walk": "Write three areas you want God to rebuild. For each one, write one responsibility that belongs to you—not to someone else.", "truth": "Luke 15 says the son rehearsed a servant speech, but the father interrupts the shame-script with robe, ring, sandals, and feast.", "prayer": "Father, I am coming home. Search me, cleanse me, and make my repentance more than emotion. Teach me to walk differently."}, {"day": 2, "title": "A New Creation", "reference": "2 Corinthians 5:14–21; Romans 6:1–14", "focus": "The Christian life is not merely behavior management. Paul describes union with Christ as death and resurrection. Old habits may still speak loudly, but they no longer hold rightful authority over the person who belongs to Christ. Your history explains some patterns; it does not get to define your identity.", "pattern": "Baptism visibly pictures the gospel pattern: burial of the old life and resurrection into newness of life.", "hidden": "Paul does not say Christ merely improves the old life; he says “new creation.” Romans 6 then uses burial and resurrection language: Christianity is not cosmetic repair but participation in Christ’s death and risen life.", "why": "God receives the repentant and rebuilds from the inside out.", "reflect": "What old sentence about myself have I mistaken for an identity?", "walk": "Name one old pattern and replace “that is just who I am” with “Christ is teaching me another way.”", "truth": "Paul does not say Christ merely improves the old life; he says “new creation.” Romans 6 then uses burial and resurrection language: Christianity is not cosmetic repair but participation in Christ’s death and risen life.", "prayer": "Jesus, teach me to live from who I am in You rather than from the habits of the person I used to be."}, {"day": 3, "title": "The Last Adam", "reference": "Genesis 3; Romans 5:12–21; 1 Corinthians 15:45–49", "focus": "Adam failed under temptation and his disobedience spread death. Christ is called the last Adam because He succeeds where Adam failed. This matters for manhood: biblical strength is not domination or self-sufficiency. It is a life submitted to God, obedient when obedience costs something.", "pattern": "Adam and Christ form one of Scripture’s clearest type-and-fulfillment patterns: one man’s disobedience and one Man’s obedience stand at the head of two humanities.", "hidden": "Romans 5 explicitly calls Adam a type of the One to come. Scripture places two representative men before us: Adam’s disobedience spreads death; Christ’s obedience opens the way of righteousness and life.", "why": "God receives the repentant and rebuilds from the inside out.", "reflect": "What atmosphere does my present obedience—or disobedience—create for the people closest to me?", "walk": "Choose one place where you have been passive. Take one humble, responsible step today.", "truth": "Romans 5 explicitly calls Adam a type of the One to come.", "prayer": "Lord, make me a person whose strength is governed by obedience to You."}, {"day": 4, "title": "Trust What You Cannot See", "reference": "Genesis 12:1–9; Hebrews 11:8–10", "focus": "God called Abraham to leave what was familiar without handing him the entire map. Faith is not certainty that circumstances will turn out exactly as we want. Faith is confidence in the One we follow when the next mile is unclear. You may not know the final outcome of your marriage or future; you can still know the next faithful step.", "pattern": "", "hidden": "Abraham receives the call before he receives the map. Hebrews highlights this on purpose: faith moved his feet before sight could explain the route.", "why": "God receives the repentant and rebuilds from the inside out.", "reflect": "Am I postponing obedience until God gives me information He has not promised to give?", "walk": "Ask, “What is the next obedient thing?” Do that one thing without demanding the whole plan.", "truth": "Abraham receives the call before he receives the map.", "prayer": "God, I do not need the whole map if I know You are leading. Give me courage for the next step."}, {"day": 5, "title": "Surrender What You Love", "reference": "Genesis 22:1–18; John 3:16", "focus": "Abraham’s journey with Isaac is difficult and profound. The beloved son carries the wood, a sacrifice is required, and God ultimately provides. Faith sometimes means placing even our deepest desires into God’s hands. Pray hard for restoration, but do not make even a good desire into an idol that determines whether you will trust God.", "pattern": "Isaac’s story prepares the biblical imagination for the beloved Son and for substitution. At Calvary, Christ Himself becomes the sacrifice God provides.", "hidden": "Genesis 22 repeatedly calls Isaac the beloved son, and Isaac carries the wood up the mountain. The story then introduces substitution: God provides the sacrifice, preparing the imagination for the beloved Son and God’s provision in Christ.", "why": "God receives the repentant and rebuilds from the inside out.", "reflect": "What am I asking God to restore—and have I quietly made that outcome a condition of my trust?", "walk": "Name your deepest desired outcome, then pray: “I ask You for this, but I belong to You even while I wait.”", "truth": "Genesis 22 repeatedly calls Isaac the beloved son, and Isaac carries the wood up the mountain.", "prayer": "Father, I place what I love into Your hands. Keep desire from becoming idolatry."}, {"day": 6, "title": "The Limp That Changes a Life", "reference": "Genesis 32:22–32", "focus": "Jacob spent years manipulating outcomes. At the Jabbok, he finally reaches a place where his own strength cannot secure the future. He leaves the encounter blessed—but limping. Some weaknesses become reminders that self-reliance is a poor savior. God may use humbling seasons to change the way a person walks.", "pattern": "", "hidden": "Jacob enters the night trying to secure blessing and leaves unable to walk as before. The changed walk becomes a bodily reminder that blessing and broken self-reliance can arrive together.", "why": "God receives the repentant and rebuilds from the inside out.", "reflect": "What disappointment or weakness has exposed the limits of my own control?", "walk": "Instead of asking only “Why did this happen?”, write an answer to “What can God form in me through this?”", "truth": "Jacob enters the night trying to secure blessing and leaves unable to walk as before.", "prayer": "God, use even my weakness to make me more dependent, truthful and teachable."}, {"day": 7, "title": "God Works in the Unfinished Story", "reference": "Genesis 50:15–21", "focus": "Joseph’s life included betrayal, injustice, false accusation, delay and eventual restoration. His conclusion was not that evil had somehow become good. It was that human evil could not overthrow the providence of God. God can take what He hates and still weave redemption through it.", "pattern": "", "hidden": "Joseph never calls evil good. He says God was able to mean good through what others meant for evil. Providence is stronger than human intent without making human sin innocent.", "why": "God receives the repentant and rebuilds from the inside out.", "reflect": "Where am I judging God’s faithfulness while the story is still unfinished?", "walk": "Write one unresolved situation. Under it write: “I do not yet know the ending, but God is not absent from the middle.”", "truth": "Joseph never calls evil good.", "prayer": "Lord, give me patience while the story is incomplete and faith that Your wisdom reaches farther than my sight."}, {"day": 8, "title": "The Wilderness Is Not Always Punishment", "reference": "Exodus 13:17–22; Deuteronomy 8:1–6", "focus": "God deliberately led Israel by a longer road because they were not ready for the shortest route. Delay can feel like abandonment, but Scripture shows that God sometimes uses the wilderness to train dependence, reveal the heart and prepare people for what comes next. Not every long road means you missed God.", "pattern": "", "hidden": "Exodus 13 says God intentionally did not choose the shortest route. Sometimes the long road is not proof that God is absent; the text says the longer way can itself be part of His guidance.", "why": "God forms covenant character through humility, truth, service, and sacrificial love.", "reflect": "What might God be developing in me that a shortcut would never produce?", "walk": "List one quality you need more than a faster answer—patience, humility, discipline, courage or trust—and practice it today.", "truth": "Exodus 13 says God intentionally did not choose the shortest route.", "prayer": "Father, if this road is longer than I hoped, do not let me waste the wilderness."}, {"day": 9, "title": "When There Is No Visible Way Forward", "reference": "Exodus 14", "focus": "Israel had Pharaoh behind and the sea ahead. God placed them where self-rescue was impossible. Moses told them the Lord would fight for them, and then God told them to move forward. Trust is not passivity. Faith can be inwardly still while outwardly obedient.", "pattern": "The Red Sea becomes a great biblical pattern of deliverance—God bringing His people out from bondage through a passage they could not create for themselves.", "hidden": "At the Red Sea, Israel could not create the path it needed. Deliverance begins with God’s action before it becomes Israel’s journey.", "why": "God forms covenant character through humility, truth, service, and sacrificial love.", "reflect": "Where do I need to stop panicking and start obeying?", "walk": "Identify one action you can take without controlling the result. Take it calmly.", "truth": "At the Red Sea, Israel could not create the path it needed.", "prayer": "Lord, fight the battles that belong to You and teach me to take the steps that belong to me."}, {"day": 10, "title": "Grace for Today", "reference": "Exodus 16; Matthew 6:25–34; John 6:32–35", "focus": "Manna trained Israel to trust God one day at a time. They could not hoard tomorrow’s dependence. Jesus later calls Himself the true Bread from heaven. Anxiety often tries to make us emotionally survive five years in a single afternoon. God gives grace in daily portions.", "pattern": "Manna is a divinely given food in the wilderness and a pattern Jesus uses to reveal Himself as the Bread of Life.", "hidden": "Manna had to be gathered fresh. Jesus later identifies Himself as the true Bread from heaven. The pattern teaches daily dependence: yesterday’s experience of grace is not a substitute for receiving Christ today.", "why": "God forms covenant character through humility, truth, service, and sacrificial love.", "reflect": "What future problem am I trying to solve emotionally today?", "walk": "Return to today: pray today, love today, work today, tell the truth today. Leave tomorrow’s manna for tomorrow.", "truth": "Manna had to be gathered fresh.", "prayer": "Jesus, be my daily bread. Keep me from borrowing trouble from days that have not arrived."}, {"day": 11, "title": "The Rock in the Wilderness", "reference": "Exodus 17:1–7; 1 Corinthians 10:1–4", "focus": "Israel thirsted where no natural supply seemed available. Paul explicitly connects the wilderness rock with Christ. A person becomes spiritually unstable when he demands that a spouse, career, child, ideology or accomplishment satisfy a thirst only God can meet. People make terrible gods because they were never created to carry that weight.", "pattern": "Paul identifies the rock with Christ, showing that God’s provision in the wilderness participates in a larger Christ-centered pattern.", "hidden": "Paul does not leave the rock as a vague symbol: 1 Corinthians 10 says “that Rock was Christ.” The wilderness provision is one of the New Testament’s clearest authorized types.", "why": "God forms covenant character through humility, truth, service, and sacrificial love.", "reflect": "Who or what have I expected to keep me emotionally full?", "walk": "Spend ten quiet minutes asking nothing from anyone. Bring your thirst directly to Christ.", "truth": "Paul does not leave the rock as a vague symbol: 1 Corinthians 10 says “that Rock was Christ.” The wilderness provision is one of the New Testament’s clearest authorized types.", "prayer": "Christ, become my source so I can love people without demanding that they save me."}, {"day": 12, "title": "Do Not Let Fear Interpret Reality", "reference": "Numbers 13:25–33; Numbers 14:1–9", "focus": "Twelve spies saw the same land and the same giants. Ten interpreted reality through fear; Joshua and Caleb interpreted it through the character and promise of God. Faith does not pretend giants are small. It refuses to make them larger than God.", "pattern": "", "hidden": "The ten spies and the two faithful spies saw the same giants. The difference was not information but interpretation: fear measured God by the giants; faith measured the giants by God.", "why": "God forms covenant character through humility, truth, service, and sacrificial love.", "reflect": "Which voice has been discipling me lately—fear or faith?", "walk": "Limit one fear-amplifying input today. Replace that time with Scripture and prayer.", "truth": "The ten spies and the two faithful spies saw the same giants.", "prayer": "God, make me sober about difficulties without becoming ruled by them."}, {"day": 13, "title": "A New Chapter Is Still a Chapter", "reference": "Joshua 1:1–9", "focus": "Moses was gone, a generation had passed, and Joshua stood at the edge of a new assignment. God did not ask Joshua to be young. He commanded him to be strong, courageous and obedient. At sixty, the question is not whether the earlier chapters were perfect. The question is what faithfulness looks like in the chapter still being written.", "pattern": "", "hidden": "Joshua 1 repeats “be strong and courageous,” but the command is anchored to God’s presence and Word. Biblical courage is confidence that God has not abandoned the assignment.", "why": "God forms covenant character through humility, truth, service, and sacrificial love.", "reflect": "Have I been speaking as though my most meaningful years must be behind me?", "walk": "Write three ways your life experience can now serve your family, church or another person.", "truth": "Joshua 1 repeats “be strong and courageous,” but the command is anchored to God’s presence and Word.", "prayer": "Lord, give me courage for the years You have actually placed in front of me."}, {"day": 14, "title": "Build Memorial Stones", "reference": "Joshua 4:1–9; Psalm 77:11–14", "focus": "God told Israel to collect stones so future generations would ask what happened there. Human beings forget deliverance quickly and remember fear vividly. Spiritual memory is a discipline. Recalling past faithfulness gives perspective to present uncertainty.", "pattern": "", "hidden": "The memorial stones in Joshua 4 were built for future questions. God designed remembrance to become intergenerational testimony: “What do these stones mean?” turns memory into discipleship.", "why": "God forms covenant character through humility, truth, service, and sacrificial love.", "reflect": "Where has God already carried me that I once thought I would never survive?", "walk": "Write five “memorial stones”—moments of provision, protection, correction, rescue or unexpected mercy.", "truth": "The memorial stones in Joshua 4 were built for future questions.", "prayer": "Father, keep me from forgetting what You have already done."}, {"day": 15, "title": "As for Me and My House", "reference": "Joshua 24:14–24", "focus": "Joshua’s declaration about his household begins with Joshua himself. Spiritual leadership is never permission to control another adult’s faith. It is the responsibility to make your own allegiance unmistakable and to let your life give weight to your words.", "pattern": "", "hidden": "Joshua’s “as for me and my house” begins with personal allegiance before household influence. Biblical leadership starts with the leader’s own worship.", "why": "Christ restores failed people without minimizing failure; grace gives forgiveness and a new direction.", "reflect": "If my family copied my actual habits rather than my advice, where would those habits lead them?", "walk": "Choose one visible practice of leadership: prayer, apology, Scripture, service or integrity. Do it without announcing it.", "truth": "Joshua’s “as for me and my house” begins with personal allegiance before household influence.", "prayer": "God, let my example become more persuasive than my lectures."}, {"day": 16, "title": "Boaz: Strength Under Control", "reference": "Ruth 2; Ruth 3:7–14", "focus": "Boaz possessed social and economic power, yet he used it to protect rather than exploit. His strength was governed by righteousness. Biblical masculinity is not harshness. It is strength disciplined by holiness, honor and concern for another person’s dignity.", "pattern": "", "hidden": "Boaz has power in the story, but he consistently uses it to protect Ruth’s dignity and provision. Scripture presents strength under covenant restraint, not strength used for exploitation.", "why": "Christ restores failed people without minimizing failure; grace gives forgiveness and a new direction.", "reflect": "Do the people closest to me feel safer because I am becoming more like Christ?", "walk": "Use your strength today to make someone else’s burden lighter rather than to get your way.", "truth": "Boaz has power in the story, but he consistently uses it to protect Ruth’s dignity and provision.", "prayer": "Lord, make me strong without making me hard, and gentle without making me weak."}, {"day": 17, "title": "David: A Faithful Person Can Still Fail", "reference": "2 Samuel 11; Psalm 51", "focus": "David’s calling did not make him immune to destructive choices. His failure shows that gifting, experience and past faithfulness never remove the need for vigilance. Psalm 51 then shows repentance without excuse-making. David asks not merely to escape consequences but to be made clean within.", "pattern": "", "hidden": "Psalm 51 shows that David’s deepest request after public failure is not image repair but an inward work: “create in me a clean heart.” Repentance aims deeper than reputation.", "why": "Christ restores failed people without minimizing failure; grace gives forgiveness and a new direction.", "reflect": "Where am I tempted to explain sin instead of confessing it?", "walk": "Confess one specific wrong to God without adding “but.” If another person is owed an apology, plan it humbly.", "truth": "Psalm 51 shows that David’s deepest request after public failure is not image repair but an inward work: “create in me a clean heart.” Repentance aims deeper than reputation.", "prayer": "Create in me a clean heart, O God, and renew a right spirit within me."}, {"day": 18, "title": "Everyone Needs a Nathan", "reference": "2 Samuel 12:1–13; Proverbs 27:5–6", "focus": "Nathan confronted a king and David eventually received the truth. Maturity includes becoming harder to flatter and easier to correct. Isolation protects self-deception. Godly friendship gives another person permission to ask the question you would rather avoid.", "pattern": "", "hidden": "Nathan’s parable gets past David’s defenses before naming his sin. Sometimes truth reaches the heart through a story before the conscience is ready for a direct accusation.", "why": "Christ restores failed people without minimizing failure; grace gives forgiveness and a new direction.", "reflect": "Who can tell me the truth without losing access to me?", "walk": "Identify one trustworthy, godly person. Invite them to ask you hard questions regularly.", "truth": "Nathan’s parable gets past David’s defenses before naming his sin.", "prayer": "Lord, protect me from the blindness that grows when nobody is allowed to challenge me."}, {"day": 19, "title": "What We Ignore Can Grow", "reference": "2 Samuel 13:21–39; 18:31–33", "focus": "David’s household shows the devastating cost of unresolved wrongs and neglected responsibilities. Avoiding conflict is not the same as making peace. Biblical peacemaking often walks directly into uncomfortable truth with humility, timing and courage.", "pattern": "", "hidden": "David’s household shows how unaddressed wrong can multiply. Silence can look like peace for a season while actually allowing disorder to deepen.", "why": "Christ restores failed people without minimizing failure; grace gives forgiveness and a new direction.", "reflect": "What conversation have I postponed because discomfort felt worse than drift?", "walk": "Write what needs to be said in one sentence. Remove blame, exaggeration and old scorekeeping before you speak.", "truth": "David’s household shows how unaddressed wrong can multiply.", "prayer": "God, give me courage to address what needs attention without anger or manipulation."}, {"day": 20, "title": "Wisdom Before Wealth", "reference": "1 Kings 3:3–14; Proverbs 4:1–13", "focus": "Solomon asked for wisdom to govern rather than merely asking for personal advantage. Wisdom is the capacity to see life from God’s perspective and choose accordingly. Anyone carrying responsibility often needs discernment more than a quick answer.", "pattern": "", "hidden": "Solomon asks for an understanding heart before riches, and God treats that request as evidence that he understands the weight of leadership. Wisdom is equipment for responsibility.", "why": "Christ restores failed people without minimizing failure; grace gives forgiveness and a new direction.", "reflect": "Do I know the difference between what is urgent and what is truly important?", "walk": "Before one decision today, pause and ask: What is true? What is loving? What is wise? What is mine to do?", "truth": "Solomon asks for an understanding heart before riches, and God treats that request as evidence that he understands the weight of leadership.", "prayer": "Give me wisdom to know when to speak, when to wait, what to confront and what to release."}, {"day": 21, "title": "The King Who Washed Feet", "reference": "John 13:1–17; Mark 10:42–45", "focus": "Jesus overturned worldly leadership by taking the posture of a servant. He did not become less authoritative by washing feet; He showed what righteous authority looks like. Anyone who wants spiritual influence in the home should begin with service, not demands for recognition.", "pattern": "The Lord taking the servant’s place anticipates the cross, where His kingship is revealed through self-giving love.", "hidden": "John 13 says Jesus knew where He came from and where He was going—then He washed feet. Secure identity produced humble service; He did not serve because He forgot His authority, but because He knew it.", "why": "Christ restores failed people without minimizing failure; grace gives forgiveness and a new direction.", "reflect": "How can I serve my family today without expecting applause?", "walk": "Do one unnoticed task that benefits your household.", "truth": "John 13 says Jesus knew where He came from and where He was going—then He washed feet.", "prayer": "Jesus, teach me to lead from the towel before I ever reach for the title."}, {"day": 22, "title": "Covenant, Not Convenience", "reference": "Genesis 2:18–25; Matthew 19:4–6", "focus": "Scripture treats marriage as covenant rather than a temporary arrangement built only on current emotion. Covenant does not require pretending problems are small. It changes the question from “Am I getting everything I want?” to “How do I honor what God has entrusted to me?”", "pattern": "", "hidden": "Genesis ends the marriage sentence with “one flesh,” and Jesus quotes that same creation text. Marriage is rooted in creation before it is discussed as a social arrangement.", "why": "God supplies truth, His Spirit, and strength to stand in temptation and conflict.", "reflect": "Where have I been evaluating marriage only through my present feelings?", "walk": "Write three covenant-minded actions you can take regardless of your spouse’s immediate response.", "truth": "Genesis ends the marriage sentence with “one flesh,” and Jesus quotes that same creation text.", "prayer": "God, teach me to honor marriage with truth, patience and faithfulness."}, {"day": 23, "title": "Love Like Christ in Marriage", "reference": "Ephesians 5:21–33", "focus": "In Ephesians 5, Christ is the pattern for covenant love, and His love is sacrificial. Spiritual leadership therefore looks like a cross before it looks like a platform. It gives, protects, sanctifies, speaks truth and seeks another person’s good. Read the passage first for what it commands you to become—not as ammunition for what someone else should do.", "pattern": "Marriage itself is used by Paul as a living sign of Christ’s covenant love for His church.", "hidden": "Ephesians 5 places marriage under Christ’s self-giving love. The controlling image is not domination but the cross: Christ loved by giving Himself.", "why": "God supplies truth, His Spirit, and strength to stand in temptation and conflict.", "reflect": "Where would sacrificial love cost me pride today?", "walk": "Choose one concrete act of service, tenderness or responsibility that asks nothing in return.", "truth": "Ephesians 5 places marriage under Christ’s self-giving love.", "prayer": "Christ, teach me to love with patience, courage and truth."}, {"day": 24, "title": "Listen Before You Answer", "reference": "James 1:19–20; Proverbs 18:13", "focus": "Quick listening and slow speech are acts of spiritual discipline. Many conflicts intensify because we listen only long enough to reload our defense. Understanding another person is not the same as agreeing with everything they say. It is making room to hear before trying to be heard.", "pattern": "", "hidden": "James places hearing before speaking. The sequence matters: spiritual maturity is not merely having the right answer but becoming slow enough to truly receive what another person is saying.", "why": "God supplies truth, His Spirit, and strength to stand in temptation and conflict.", "reflect": "When someone I love speaks about hurt, do I become curious—or immediately defensive?", "walk": "In one conversation today, do not interrupt. Summarize what you heard before offering your perspective.", "truth": "James places hearing before speaking.", "prayer": "Lord, make me quick to hear, slow to speak and slow to anger."}, {"day": 25, "title": "The Power of a Soft Answer", "reference": "Proverbs 15:1–4; Proverbs 12:18", "focus": "Words can function like medicine or weapons. A gentle answer does not mean cowardice or avoiding truth. It means truth is no longer carried by contempt. You cannot control another person’s tone, but you can refuse to add fuel to a fire.", "pattern": "", "hidden": "Proverbs says a soft answer can turn away wrath. Gentleness is not the absence of truth; it is truth delivered without adding unnecessary fire.", "why": "God supplies truth, His Spirit, and strength to stand in temptation and conflict.", "reflect": "In conflict, am I trying to solve the problem or win the moment?", "walk": "Before your next difficult reply, lower your volume and shorten your sentence.", "truth": "Proverbs says a soft answer can turn away wrath.", "prayer": "God, govern my tongue. Let my words heal more than they wound."}, {"day": 26, "title": "Forgiveness and Trust", "reference": "Colossians 3:12–14; Luke 17:3–4", "focus": "Believers are called to cultivate forgiving hearts, but trust often rebuilds through time, consistency and visible change. An apology cannot be used to demand immediate access to someone’s confidence. Let repentance become a pattern sturdy enough for another person to observe.", "pattern": "", "hidden": "Colossians puts forgiveness inside a larger wardrobe of compassion, humility, meekness, patience, and love. Forgiveness is part of becoming clothed with Christlike character.", "why": "God supplies truth, His Spirit, and strength to stand in temptation and conflict.", "reflect": "Am I asking someone to trust my words faster than my life has earned?", "walk": "Choose one promise you can keep consistently for the next thirty days. Make it small enough to be real.", "truth": "Colossians puts forgiveness inside a larger wardrobe of compassion, humility, meekness, patience, and love.", "prayer": "Lord, teach me to forgive freely and rebuild trust patiently."}, {"day": 27, "title": "Love Keeps No Scoreboard", "reference": "1 Corinthians 13:1–8", "focus": "Read Paul’s description of love as a mirror, not a weapon. Replace the word “love” with your own name and notice where the sentence becomes difficult: patient, kind, not self-seeking, not easily provoked, not keeping a ledger of wrongs. Mature love stops collecting ammunition.", "pattern": "", "hidden": "In 1 Corinthians 13, love “keeps no record of wrongs” in the sense of not maintaining an accounting ledger for retaliation. Love does not erase wisdom, but it refuses to turn relationship into bookkeeping for revenge.", "why": "God supplies truth, His Spirit, and strength to stand in temptation and conflict.", "reflect": "Which phrase in 1 Corinthians 13 exposes me most?", "walk": "Practice that one quality deliberately today.", "truth": "In 1 Corinthians 13, love “keeps no record of wrongs” in the sense of not maintaining an accounting ledger for retaliation.", "prayer": "Spirit of God, form in me the kind of love I cannot manufacture by willpower alone."}, {"day": 28, "title": "Fight for the Marriage, Not for Control", "reference": "Romans 12:9–21", "focus": "There is a profound difference between fighting for a marriage and trying to control another person. Romans 12 repeatedly brings responsibility back to the believer: sincere love, humility, patience, peace as far as it depends on you. Personal obedience does not require another person’s cooperation.", "pattern": "", "hidden": "Romans 12 moves from sincere love to blessing persecutors, refusing vengeance, feeding enemies, and overcoming evil with good. The chapter defines victory very differently from domination.", "why": "God supplies truth, His Spirit, and strength to stand in temptation and conflict.", "reflect": "What belongs to my responsibility, and what am I trying to control that belongs to someone else?", "walk": "Draw two columns: “Mine to obey” and “Mine to release.” Put the right things in each.", "truth": "Romans 12 moves from sincere love to blessing persecutors, refusing vengeance, feeding enemies, and overcoming evil with good.", "prayer": "God, make me faithful without making me controlling. Teach me to entrust outcomes to You."}, {"day": 29, "title": "Peter’s Worst Night", "reference": "Luke 22:31–34, 54–62", "focus": "Peter confidently promised loyalty and then denied Jesus three times. Jesus knew the collapse was coming—and still spoke of Peter’s future usefulness: “when you have turned again, strengthen your brothers.” Christ can see ministry on the far side of a failure we think has ended everything.", "pattern": "", "hidden": "Peter denied Jesus beside a charcoal fire; John 21 places his restoration beside another charcoal fire. John uses the same rare detail to bring Peter back to the place of failure—this time to be recommissioned.", "why": "God remains King above every human system and ruler.", "reflect": "What failure have I treated as a permanent identity?", "walk": "Write what the failure taught you that success never could.", "truth": "Peter denied Jesus beside a charcoal fire; John 21 places his restoration beside another charcoal fire.", "prayer": "Jesus, do not let my failure be wasted. Turn it into humility, wisdom and compassion."}, {"day": 30, "title": "Peter Is Recommissioned", "reference": "John 21:1–19", "focus": "Jesus meets Peter by a charcoal fire, revisits the question of love, and then gives him responsibility: feed My sheep. Grace does more than cancel debt; it can restore calling. Recommissioning does not pretend the denial never happened. It proves that failure does not get the final word.", "pattern": "The risen Christ restores the disciple who denied Him, displaying resurrection not only as an event but as the power of renewed vocation.", "hidden": "Jesus asks Peter three times about love after Peter’s three denials. Restoration is not pretending the failure never happened; Christ revisits the wound and then gives Peter work to do: “feed My sheep.”", "why": "God remains King above every human system and ruler.", "reflect": "If Christ is not finished with me, what faithful work might still be in front of me?", "walk": "Do one constructive thing today that shame has made you avoid.", "truth": "Jesus asks Peter three times about love after Peter’s three denials.", "prayer": "Lord, thank You that restoration includes purpose. Show me where to serve faithfully now."}, {"day": 31, "title": "No Condemnation", "reference": "Romans 8:1–17", "focus": "Conviction and condemnation move in opposite directions. Conviction names sin and draws us toward God for change. Condemnation tells us there is no future and urges us to hide. Romans 8 begins with no condemnation for those in Christ and moves into life by the Spirit.", "pattern": "", "hidden": "Romans 8 begins with no condemnation and moves immediately into life in the Spirit. Grace is not permission to remain unchanged; it is the new ground from which transformation becomes possible.", "why": "God remains King above every human system and ruler.", "reflect": "Does the voice I am listening to lead me toward repentance—or hopelessness?", "walk": "When a condemning thought appears, answer it with Romans 8:1 and then take the next obedient step.", "truth": "Romans 8 begins with no condemnation and moves immediately into life in the Spirit.", "prayer": "Father, make me sensitive to conviction and resistant to hopeless condemnation."}, {"day": 32, "title": "Godly Sorrow Changes Direction", "reference": "2 Corinthians 7:8–11", "focus": "Paul distinguishes godly sorrow from worldly sorrow. Worldly sorrow can remain self-centered: embarrassment, loss, consequences, wounded pride. Godly sorrow produces earnestness and change. The evidence of repentance is not the intensity of a feeling but the direction of a life.", "pattern": "", "hidden": "Paul distinguishes worldly sorrow from godly sorrow by its fruit. Feeling terrible is not the same as repentance; godly sorrow produces a changed direction.", "why": "God remains King above every human system and ruler.", "reflect": "What has my sorrow actually changed?", "walk": "Identify one habit, boundary or practice that should look different because you repented.", "truth": "Paul distinguishes worldly sorrow from godly sorrow by its fruit.", "prayer": "God, make my sorrow fruitful. Let it produce a changed direction."}, {"day": 33, "title": "Repentance Becomes Tangible", "reference": "Luke 19:1–10", "focus": "Zacchaeus’s encounter with Jesus reached his wallet and his relationships. He wanted to repair what he had harmed. Biblical repentance moves beyond private emotion into visible integrity wherever repair is possible.", "pattern": "", "hidden": "Zacchaeus’ repentance becomes measurable: he speaks about restitution and generosity. Grace reaches the heart and then shows up in the wallet.", "why": "God remains King above every human system and ruler.", "reflect": "Is there something I can make right instead of merely feeling bad about it?", "walk": "Consider an apology, repayment, corrected statement or kept promise. Do what is wise and possible without demanding a response.", "truth": "Zacchaeus’ repentance becomes measurable: he speaks about restitution and generosity.", "prayer": "Jesus, let my repentance grow hands and feet."}, {"day": 34, "title": "God Can Restore Wasted Years", "reference": "Joel 2:12–27", "focus": "Joel’s promise of restoration does not mean yesterday literally returns. God does something more mysterious: He can bring such fruit from surrendered years that the story of loss is no longer the only story being told. Do not waste the next decade grieving the last one.", "pattern": "", "hidden": "Joel speaks to people who have lost years to devastation, yet God’s promise is restoration rather than mere survival. The locust years become part of testimony, not the final definition of the future.", "why": "God remains King above every human system and ruler.", "reflect": "What am I still mourning that I now need to place in God’s hands?", "walk": "Write a sentence beginning, “With the years I have left, by God’s grace I will…”", "truth": "Joel speaks to people who have lost years to devastation, yet God’s promise is restoration rather than mere survival.", "prayer": "Lord, I cannot retrieve yesterday. I give You the years that remain."}, {"day": 35, "title": "Beauty From Ashes", "reference": "Isaiah 61:1–7; Luke 4:16–21", "focus": "Jesus read Isaiah’s promise of good news, liberty and restoration and applied it to His own mission. God specializes in places human beings label ruined. Redemption does not always rebuild the structure exactly as it was, but it never treats ashes as the end of the story.", "pattern": "Isaiah’s Servant-shaped promise finds its fulfillment in Jesus’ announced mission.", "hidden": "Jesus reads Isaiah 61 in Nazareth and says the Scripture is fulfilled in their hearing. He does not merely teach the promise; He presents Himself as its fulfillment.", "why": "God remains King above every human system and ruler.", "reflect": "Can I surrender not only the ashes but also my preferred blueprint for restoration?", "walk": "Pray for restoration without dictating the form it must take.", "truth": "Jesus reads Isaiah 61 in Nazareth and says the Scripture is fulfilled in their hearing.", "prayer": "Jesus, bring Your kind of restoration into the places I cannot rebuild by force."}, {"day": 36, "title": "Temptation Starts Before the Action", "reference": "James 1:12–18", "focus": "James traces temptation from desire to conception to sin to death. Wise people fight early. Waiting until temptation is fully grown and then trying to become heroic is poor strategy. Holiness often looks like building boundaries before desire becomes a crisis.", "pattern": "", "hidden": "James traces temptation backward from action to desire. The battle is usually older than the behavior we finally notice.", "why": "God rebuilds patiently and calls His people to focused endurance.", "reflect": "What situations make me predictably weaker?", "walk": "Create one boundary today around time, technology, money, anger, secrecy or another vulnerable place.", "truth": "James traces temptation backward from action to desire.", "prayer": "God, give me wisdom to flee early instead of negotiating late."}, {"day": 37, "title": "Jesus in the Wilderness", "reference": "Matthew 4:1–11", "focus": "Jesus repeatedly answered temptation with “It is written.” Scripture was not decoration; it was truth brought into battle. A Bible you admire but never internalize cannot readily answer the lie that arrives at 2 a.m. Memorization turns Scripture into language the Spirit can bring to mind.", "pattern": "Jesus succeeds in the wilderness where Israel repeatedly failed, fulfilling the pattern of the faithful Son.", "hidden": "In the wilderness Jesus answers temptation from Deuteronomy—the very book that interprets Israel’s wilderness years. Where Israel failed in the wilderness, the faithful Son stands firm.", "why": "God rebuilds patiently and calls His people to focused endurance.", "reflect": "Which lie do I hear most often when I am vulnerable?", "walk": "Begin memorizing Proverbs 3:5–6, Romans 8:1, Philippians 4:6–8 or another passage that directly answers that lie.", "truth": "In the wilderness Jesus answers temptation from Deuteronomy—the very book that interprets Israel’s wilderness years.", "prayer": "Lord, put Your Word deep in me so truth is available when temptation speaks."}, {"day": 38, "title": "Put On the Armor", "reference": "Ephesians 6:10–20", "focus": "Paul’s armor combines belief, character and practice: truth, righteousness, the gospel of peace, faith, salvation, the Word and prayer. Spiritual warfare is often less spectacular than people imagine. It looks like refusing lies, resisting temptation, praying steadily and remaining faithful under pressure.", "pattern": "", "hidden": "Paul’s armor imagery is almost entirely defensive and relational to God: truth, righteousness, gospel peace, faith, salvation, the Word, and prayer. Spiritual warfare begins with standing in what God supplies.", "why": "God rebuilds patiently and calls His people to focused endurance.", "reflect": "Which piece of this armor have I been neglecting?", "walk": "Read Ephesians 6 slowly and pray each piece over your day.", "truth": "Paul’s armor imagery is almost entirely defensive and relational to God: truth, righteousness, gospel peace, faith, salvation, the Word, and prayer.", "prayer": "God, make me alert without making me fearful and spiritually armed without becoming theatrical."}, {"day": 39, "title": "Take Thoughts Captive", "reference": "2 Corinthians 10:3–5; Philippians 4:4–9", "focus": "Not every thought deserves a chair at your table. Some thoughts should be examined and rejected. “I will never change,” “everything is hopeless,” and “God has left me” feel powerful, but feelings do not convert lies into truth. Christian maturity includes learning to interrogate thoughts by Scripture.", "pattern": "", "hidden": "2 Corinthians 10 locates a major battlefield in arguments and thoughts. Captivity here is bringing thought under the obedience of Christ rather than letting thought become lord.", "why": "God rebuilds patiently and calls His people to focused endurance.", "reflect": "Which recurring thought has been acting like a fact without being tested?", "walk": "Write the thought, then write one biblical truth that corrects it.", "truth": "2 Corinthians 10 locates a major battlefield in arguments and thoughts.", "prayer": "Lord, renew my mind and teach me to refuse mental agreements with lies."}, {"day": 40, "title": "Guard the Gate of Your Heart", "reference": "Proverbs 4:20–27", "focus": "What repeatedly enters the mind shapes the inner life, and the inner life eventually directs the feet. News, outrage, social media, entertainment and arguments all disciple us in some direction. Being informed is not the same as being saturated.", "pattern": "", "hidden": "Proverbs 4 says to guard the heart because the issues of life flow from it. Biblical guarding is not emotional shutdown; it is careful stewardship of the inner source from which words and choices emerge.", "why": "God rebuilds patiently and calls His people to focused endurance.", "reflect": "What input leaves me consistently more fearful, angry, lustful or contemptuous?", "walk": "Fast from one unhealthy input for twenty-four hours and replace it with something that produces wisdom and peace.", "truth": "Proverbs 4 says to guard the heart because the issues of life flow from it.", "prayer": "God, help me guard my heart without withdrawing from the world You call me to serve."}, {"day": 41, "title": "Be Angry and Sin Not", "reference": "Ephesians 4:25–32", "focus": "Scripture does not pretend anger never appears. It refuses to let anger become a permanent resident. Unprocessed anger hardens into bitterness, and bitterness eventually interprets everything through injury. Forgiven people should become progressively less willing to keep old offenses alive for leverage.", "pattern": "", "hidden": "Ephesians allows anger but gives it a clock: do not let the sun go down on it. The text treats unresolved anger as spiritually dangerous because it can become an opening for further evil.", "why": "God rebuilds patiently and calls His people to focused endurance.", "reflect": "What old offense do I rehearse because it gives me a sense of moral advantage?", "walk": "Stop retelling one grievance today. If action is still needed, address it directly rather than rehearsing it indirectly.", "truth": "Ephesians allows anger but gives it a clock: do not let the sun go down on it.", "prayer": "God, free me from anger that has outlived its usefulness."}, {"day": 42, "title": "The Fruit Test", "reference": "Galatians 5:16–26", "focus": "Love, joy, peace, patience, kindness, goodness, faithfulness, gentleness and self-control are evidence of the Spirit’s work. Spiritual growth cannot be measured only by how much doctrine we know. A mature believer should increasingly resemble the character of Christ in ordinary relationships.", "pattern": "", "hidden": "Galatians calls love, joy, peace, patience and the rest “fruit”—organic evidence of the Spirit’s life. Fruit grows from abiding life, not from stapling religious behavior onto an unchanged root.", "why": "God rebuilds patiently and calls His people to focused endurance.", "reflect": "Which fruit is least visible in me at home?", "walk": "Choose one fruit and practice it intentionally in three small interactions today.", "truth": "Galatians calls love, joy, peace, patience and the rest “fruit”—organic evidence of the Spirit’s life.", "prayer": "Holy Spirit, grow in me what effort alone cannot manufacture."}, {"day": 43, "title": "God Rules Above Human Kingdoms", "reference": "Daniel 2:19–23; Daniel 4:34–37", "focus": "Daniel lived under massive empires and watched rulers rise and fall. Scripture does not treat government as irrelevant, but it refuses to treat government as ultimate. God’s sovereignty stands above administrations, nations and historical upheaval. That conviction should create steadiness rather than apathy.", "pattern": "", "hidden": "Daniel 2 says God removes kings and sets up kings. Political power matters, but Scripture places it under a larger sovereignty that no election, empire, or ruler can escape.", "why": "God forms legacy through abiding, obedience, remembrance, and daily allegiance.", "reflect": "Does political uncertainty shake me more deeply than the sovereignty of God steadies me?", "walk": "Pray about one national concern, then deliberately entrust the outcome to God before returning to your responsibilities.", "truth": "Daniel 2 says God removes kings and sets up kings.", "prayer": "King of heaven, keep me engaged without letting earthly power become the source of my hope or fear."}, {"day": 44, "title": "Nebuchadnezzar Learns Who Is King", "reference": "Daniel 4", "focus": "Nebuchadnezzar possessed extraordinary power and still learned that human authority is temporary and accountable. Scripture consistently relativizes political power beneath divine rule. No ruler deserves the worship that belongs to God, and no ruler possesses the final authority over history.", "pattern": "", "hidden": "Nebuchadnezzar’s restoration begins when he lifts his eyes toward heaven. The proud king recovers perspective when he stops treating himself as the highest reality.", "why": "God forms legacy through abiding, obedience, remembrance, and daily allegiance.", "reflect": "Have I magnified a political figure—positively or negatively—beyond the limits Scripture gives human rulers?", "walk": "Name one leader you strongly react to and pray for that person’s wisdom, justice and accountability before God.", "truth": "Nebuchadnezzar’s restoration begins when he lifts his eyes toward heaven.", "prayer": "Lord, keep my heart free from both political idolatry and political terror."}, {"day": 45, "title": "Daniel Served Without Becoming Babylon", "reference": "Daniel 1", "focus": "Daniel learned Babylon’s language and served within its government, yet he established boundaries when assimilation would violate faithfulness to God. Christians can participate constructively in societies that do not fully reflect the kingdom of God. Participation is not the same as ultimate allegiance.", "pattern": "", "hidden": "Daniel 1 shows four young men learning Babylon’s language and serving in Babylon’s system without letting Babylon rename their deepest allegiance. Faithfulness can live inside a culture without being absorbed by it.", "why": "God forms legacy through abiding, obedience, remembrance, and daily allegiance.", "reflect": "Where do I need wisdom to participate without being absorbed?", "walk": "Write one conviction you will not compromise and one way you can still serve people who disagree with you.", "truth": "Daniel 1 shows four young men learning Babylon’s language and serving in Babylon’s system without letting Babylon rename their deepest allegiance.", "prayer": "God, make me useful in the world without letting the world own my conscience."}, {"day": 46, "title": "Pray for Leaders", "reference": "1 Timothy 2:1–6", "focus": "Paul directs believers to pray for kings and all in authority. The command did not depend on liking those leaders. Prayer for public officials is not the same as endorsing them. It is a Christian act of seeking peace, justice, restraint of evil and conditions in which people may live godly lives.", "pattern": "", "hidden": "1 Timothy tells believers to pray for kings even when the rulers themselves are not believers. Prayer for government is not endorsement; it is obedience aimed at peaceful, godly life and the spread of truth.", "why": "God forms legacy through abiding, obedience, remembrance, and daily allegiance.", "reflect": "Do I talk about leaders more than I pray for them?", "walk": "Pray for local, state and national leaders by office. Ask for wisdom, justice, humility and protection of people—not partisan advantage.", "truth": "1 Timothy tells believers to pray for kings even when the rulers themselves are not believers.", "prayer": "God, grant those with authority wisdom and restrain the misuse of power."}, {"day": 47, "title": "Render to Caesar—Remember God’s Image", "reference": "Matthew 22:15–22; Genesis 1:26–27", "focus": "Jesus points to Caesar’s image on the coin and acknowledges a legitimate civic claim. But Genesis tells us whose image is stamped upon human beings. Governments may have limited claims on taxes, law and civic life; God has the deepest claim upon the person. That keeps civil obligations in their proper place.", "pattern": "The image on the coin creates a powerful contrast with the divine image borne by every human person.", "hidden": "Jesus asks whose image is on the coin, then says to give God what belongs to God. Genesis tells us whose image is on you. Caesar may stamp currency; God stamps humanity.", "why": "God forms legacy through abiding, obedience, remembrance, and daily allegiance.", "reflect": "Have I confused a legitimate civic loyalty with the allegiance that belongs only to God?", "walk": "Before consuming political media today, read a Gospel passage and let Christ set the emotional temperature.", "truth": "Jesus asks whose image is on the coin, then says to give God what belongs to God.", "prayer": "Jesus, keep every earthly allegiance beneath my allegiance to You."}, {"day": 48, "title": "Do Not Let Politics Disciple Your Heart", "reference": "Psalm 1; Psalm 146:1–5", "focus": "“Put not your trust in princes” is not a command to ignore public life. It is a warning against making political power carry the weight of ultimate hope. Examine the fruit your political habits produce. Information that continually makes you contemptuous, panicked or obsessed may be shaping you more than you realize.", "pattern": "", "hidden": "Psalm 146 warns not to put ultimate trust in princes because their plans perish. Political engagement may be real, but political salvation is a category Scripture refuses to grant to human rulers.", "why": "God forms legacy through abiding, obedience, remembrance, and daily allegiance.", "reflect": "What is politics producing in my character?", "walk": "Set a defined time for news today. Use the time you recover for prayer, family or service.", "truth": "Psalm 146 warns not to put ultimate trust in princes because their plans perish.", "prayer": "God, make me informed without making me inflamed, and engaged without becoming consumed."}, {"day": 49, "title": "Your Citizenship Is Higher", "reference": "Philippians 3:17–21; Hebrews 11:13–16", "focus": "Believers belong to earthly communities and carry real civic responsibilities, yet Paul also locates Christian citizenship in heaven. No party, nation or ideology can completely contain Christian allegiance. Christ remains King whether an election goes the way you wanted or not.", "pattern": "", "hidden": "Philippians was written into a Roman world proud of citizenship, yet Paul tells believers their decisive citizenship is in heaven. Earthly loyalty is therefore never allowed to become ultimate identity.", "why": "God forms legacy through abiding, obedience, remembrance, and daily allegiance.", "reflect": "Can people see my allegiance to Christ more clearly than my allegiance to a political tribe?", "walk": "Do one act of kindness for someone whose politics you do not know—and do not ask.", "truth": "Philippians was written into a Roman world proud of citizenship, yet Paul tells believers their decisive citizenship is in heaven.", "prayer": "King Jesus, make Your kingdom the deepest source of my identity and hope."}, {"day": 50, "title": "Survey the Ruins", "reference": "Nehemiah 1–2", "focus": "Nehemiah did not pretend Jerusalem was fine. He wept, prayed, inspected the damage, planned and then acted. Rebuilding begins with truthful assessment. Denial feels easier in the short term, but it leaves broken walls exposed.", "pattern": "", "hidden": "Nehemiah inspects the ruins at night before announcing a plan. Faith does not require pretending damage is smaller than it is; honest assessment can be the first act of rebuilding.", "why": "God is trustworthy beyond what can be seen.", "reflect": "What area of my life needs an honest inspection rather than another explanation?", "walk": "Assess five areas from 1–10: walk with God, marriage, family, health, finances. Choose the lowest one for one concrete next step.", "truth": "Nehemiah inspects the ruins at night before announcing a plan.", "prayer": "God, give me courage to see accurately and hope to rebuild patiently."}, {"day": 51, "title": "Build and Stay Alert", "reference": "Nehemiah 4", "focus": "Nehemiah’s workers rebuilt while remaining watchful. Some seasons require construction and spiritual resistance at the same time. You do not have to wait for all opposition, temptation or anxiety to disappear before beginning good work.", "pattern": "", "hidden": "Nehemiah’s builders worked with tools and weapons nearby. The image holds two responsibilities together: build what God gave you to build and stay awake to what threatens the work.", "why": "God is trustworthy beyond what can be seen.", "reflect": "What good work have I postponed until life feels easier?", "walk": "Give thirty focused minutes to rebuilding something important today.", "truth": "Nehemiah’s builders worked with tools and weapons nearby.", "prayer": "Lord, strengthen my hands for the work and keep my heart alert."}, {"day": 52, "title": "Do Not Come Down From the Wall", "reference": "Nehemiah 6:1–9", "focus": "Nehemiah’s opponents repeatedly tried to pull him away from his assignment. His answer was essentially: I am doing a great work and cannot come down. Not every invitation deserves attention. Not every criticism deserves a response. Mature people learn which distractions cost too much.", "pattern": "", "hidden": "Nehemiah’s enemies repeatedly try to pull him off the wall into distraction. His answer—“I am doing a great work and cannot come down”—is a theology of focused stewardship.", "why": "God is trustworthy beyond what can be seen.", "reflect": "What repeatedly pulls me off the wall I am supposed to be rebuilding?", "walk": "Say no to one unnecessary distraction today.", "truth": "Nehemiah’s enemies repeatedly try to pull him off the wall into distraction.", "prayer": "God, give me focus for the work You have actually given me."}, {"day": 53, "title": "Become Planted", "reference": "Psalm 1", "focus": "Psalm 1 pictures the faithful person as a tree planted beside water. Rooted people are not controlled by every emotional wind. Roots grow through repeated rhythms: Scripture, prayer, fellowship, integrity, rest and service. Consistency becomes shelter for the people around you.", "pattern": "", "hidden": "Psalm 1 contrasts a tree planted by streams with chaff driven by wind. Stability in Scripture comes from rootedness, not from the absence of weather.", "why": "God is trustworthy beyond what can be seen.", "reflect": "Which spiritual rhythm is missing enough that my roots are shallow?", "walk": "Choose a realistic daily time and place for Scripture for the next seven days.", "truth": "Psalm 1 contrasts a tree planted by streams with chaff driven by wind.", "prayer": "Plant me deeply in Your truth until steadiness becomes part of my character."}, {"day": 54, "title": "Build on the Rock", "reference": "Matthew 7:24–27", "focus": "Both builders hear Jesus. Only one does what He says. The difference between sand and rock is not exposure to truth but obedience to truth. Bible knowledge can create false confidence if nothing changes below the neck.", "pattern": "The house on rock is Jesus’ own picture of a life established by hearing and doing His words.", "hidden": "Jesus ends the Sermon on the Mount with two builders who both hear His words; the difference is practice. The rock is revealed in obedience, not admiration.", "why": "God is trustworthy beyond what can be seen.", "reflect": "What have I learned from Scripture recently that I have not yet obeyed?", "walk": "Turn one piece of knowledge into one behavior today.", "truth": "Jesus ends the Sermon on the Mount with two builders who both hear His words; the difference is practice.", "prayer": "Jesus, keep me from mistaking hearing for following."}, {"day": 55, "title": "Lay Aside the Weight", "reference": "Hebrews 12:1–3", "focus": "Hebrews distinguishes between sin and weight. Some things may not be obviously sinful and still make faithful running harder. Resentment, endless distraction, pride, unhealthy relationships or the need to be right can become unnecessary weight.", "pattern": "", "hidden": "Hebrews distinguishes sin from “every weight.” Some things are not inherently sinful but still make the race harder. Maturity asks not only “Is this wrong?” but “Does this help me run?”", "why": "God is trustworthy beyond what can be seen.", "reflect": "What am I carrying that makes obedience harder?", "walk": "Name one weight and decide what putting it down looks like in practice.", "truth": "Hebrews distinguishes sin from “every weight.” Some things are not inherently sinful but still make the race harder.", "prayer": "Lord, lighten me for the race You have set before me."}, {"day": 56, "title": "Finish Better Than You Started", "reference": "2 Timothy 4:6–8", "focus": "Paul’s final testimony is not that he never failed. It is that he fought, finished and kept the faith. Finishing faithfully is a worthy ambition for every Christian. The remaining years do not need to be the easiest years to become the most faithful years.", "pattern": "", "hidden": "Paul’s final testimony is athletic and covenantal: fought, finished, kept the faith. Finishing well is not about a flawless record but sustained faithfulness to the end.", "why": "God is trustworthy beyond what can be seen.", "reflect": "What would “keeping the faith” look like in my ordinary life this year?", "walk": "Write a one-paragraph description of the person you want to be ten years from now if God grants those years.", "truth": "Paul’s final testimony is athletic and covenantal: fought, finished, kept the faith.", "prayer": "God, give me endurance to finish with faith, humility and love."}, {"day": 57, "title": "What Will Your Family Remember?", "reference": "Deuteronomy 6:4–9; Psalm 78:1–7", "focus": "Scripture expects faith to be spoken about in the ordinary flow of life and carried across generations. Legacy is not merely what a person leaves to people. It is what he leaves inside them. Your family does not need a perfect patriarch. They can be deeply helped by a person who prays, apologizes, serves and keeps returning to God.", "pattern": "", "hidden": "Deuteronomy 6 puts God’s words first on the parent’s own heart and then in the child’s hearing. Legacy begins with embodied truth before it becomes family instruction.", "why": "God is trustworthy beyond what can be seen.", "reflect": "What do I want my family to say I became?", "walk": "Complete: “From this point forward, I want my family to know me as someone who…”", "truth": "Deuteronomy 6 puts God’s words first on the parent’s own heart and then in the child’s hearing.", "prayer": "Lord, make my remaining years a blessing that outlives me."}, {"day": 58, "title": "The Father Who Ran", "reference": "Luke 15:11–32", "focus": "Return to the story where this journey began, but now watch the father. He sees, runs, embraces and restores. After weeks of concentrating on becoming a more faithful spouse, family member and disciple, remember the foundation: before you lead anyone, you are a child who needs the Father’s grace.", "pattern": "", "hidden": "The father in Luke 15 sees the son “a great way off” and runs. The story makes the father the surprising mover toward reconciliation before the son can complete his prepared speech.", "why": "God is trustworthy beyond what can be seen.", "reflect": "Am I obeying to earn love—or obeying because I have been loved?", "walk": "Spend ten minutes with God without a performance goal. Simply thank Him for grace.", "truth": "The father in Luke 15 sees the son “a great way off” and runs.", "prayer": "Father, keep sonship beneath everything I do for You."}, {"day": 59, "title": "Abide", "reference": "John 15:1–17", "focus": "Jesus does not command branches to manufacture fruit by anxiety. He commands them to abide. Christian transformation is relational before it is mechanical. Discipline matters, but discipline is meant to keep us near Christ, not to create an impressive spiritual scorecard.", "pattern": "Vine and branches is Jesus’ own image for dependent, living union with Him.", "hidden": "Jesus does not command branches to manufacture fruit; He commands them to abide. Fruitfulness is the consequence of living connection before it is the achievement of effort.", "why": "God is trustworthy beyond what can be seen.", "reflect": "Have spiritual practices become tasks I complete rather than places I meet Christ?", "walk": "Read John 15 slowly with no timer. Stop wherever a phrase arrests your attention.", "truth": "Jesus does not command branches to manufacture fruit; He commands them to abide.", "prayer": "Jesus, keep me near You. Let fruit grow from communion rather than performance."}, {"day": 60, "title": "Choose This Day", "reference": "Joshua 24:14–15; Hebrews 12:1–3; Philippians 1:6", "focus": "You cannot relive childhood, your twenties, your first years of marriage or yesterday morning. But you possess today. Scripture repeatedly shows God doing remarkable things through people who finally give Him the day in front of them. The goal is not to stare backward at the person you were but to keep looking toward Christ and become the person He is forming now.", "pattern": "", "hidden": "Joshua 24 and Hebrews 12 both force a decision about allegiance and endurance. Faithfulness is renewed in actual choices, not merely admired as an idea.", "why": "God is trustworthy beyond what can be seen.", "reflect": "What does surrender look like today—not someday?", "walk": "Write and sign your own covenant of surrender: faith, marriage, family, integrity, thought life, service and the years that remain.", "truth": "Joshua 24 and Hebrews 12 both force a decision about allegiance and endurance.", "prayer": "Lord Jesus, the remainder of my life belongs to You. Make me truthful, humble, strong, gentle, faithful and teachable. Teach me to love my family, repent quickly, forgive freely, keep Your Word as my foundation and finish following You. Amen."}, {"day": 61, "title": "Walk by Faith, Not by Sight", "reference": "2 Corinthians 4:16–18; 5:1–7", "focus": "Faith does not mean pretending visible circumstances are unreal. Paul calls believers to weigh the visible against the eternal. What can be seen is temporary; what cannot be seen is not therefore imaginary. A mature believer learns to make decisions from what God has said, not only from what today seems to prove.", "pattern": "", "hidden": "Paul contrasts what is seen and temporary with what is unseen and eternal. Faith is not denial of visible facts; it is refusal to let temporary facts define ultimate reality.", "why": "God welcomes dependent prayer as Father and Provider.", "reflect": "What visible circumstance has been defining reality for me more than God’s Word?", "walk": "Name the circumstance plainly, then write beside it one eternal truth Scripture gives you.", "truth": "Paul contrasts what is seen and temporary with what is unseen and eternal.", "prayer": "Lord, train my eyes to see beyond the moment and my feet to walk by faith."}, {"day": 62, "title": "Faith Comes by Hearing", "reference": "Romans 10:8–17; Psalm 119:9–16", "focus": "Faith is strengthened as the Word of Christ is heard, received, remembered, and obeyed. You cannot control every thought that enters your mind, but you can decide what voice receives repeated access. Scripture stored in the heart becomes a reservoir for seasons when emotions become unreliable.", "pattern": "", "hidden": "Romans says faith comes by hearing and hearing by the Word concerning Christ. Biblical faith is fed by revelation, not by trying to generate stronger feelings.", "why": "God welcomes dependent prayer as Father and Provider.", "reflect": "Which voice has had the most repetition in my life this week?", "walk": "Read today’s passage aloud twice. Choose one verse to memorize.", "truth": "Romans says faith comes by hearing and hearing by the Word concerning Christ.", "prayer": "God, let Your Word become louder in me than fear, accusation, or confusion."}, {"day": 63, "title": "The God Who Sees in Secret", "reference": "Genesis 16:7–14; Matthew 6:1–6", "focus": "Hagar names the Lord as the God who sees her. Jesus later teaches that the Father sees what is done in secret. Much of a person’s true spiritual life is built where nobody applauds. Private prayer, unseen integrity, quiet service, and hidden obedience form the person others eventually experience.", "pattern": "The God who saw Hagar in the wilderness is the same Father Jesus says sees in secret. Scripture repeatedly reveals a God attentive to the overlooked and unseen.", "hidden": "Hagar is the first person in Scripture recorded as giving God a name: El Roi, “the God who sees me.” The overlooked woman discovers that heaven has not overlooked her.", "why": "God welcomes dependent prayer as Father and Provider.", "reflect": "Who am I when no one is watching?", "walk": "Do one act of obedience today that nobody else needs to know about.", "truth": "Hagar is the first person in Scripture recorded as giving God a name: El Roi, “the God who sees me.” The overlooked woman discovers that heaven has not overlooked her.", "prayer": "Father, make my private life worthy of Your gaze."}, {"day": 64, "title": "Wait on the Lord", "reference": "Psalm 27; Isaiah 40:27–31", "focus": "Waiting on God is not spiritual inactivity. Biblical waiting is hopeful dependence: continuing to seek, obey, pray, and trust while the answer is delayed. Isaiah does not promise that waiting people never become tired; he promises renewed strength from the Lord.", "pattern": "", "hidden": "Isaiah says those who wait on the LORD renew strength. Waiting is not passive resignation but hope stretched toward God as the source of renewed capacity.", "why": "God welcomes dependent prayer as Father and Provider.", "reflect": "Where has delay tempted me to abandon trust?", "walk": "Instead of asking God only to shorten the wait, ask what faithfulness looks like inside it.", "truth": "Isaiah says those who wait on the LORD renew strength.", "prayer": "Lord, renew my strength while I wait. Keep delay from turning into unbelief."}, {"day": 65, "title": "Faith That Works", "reference": "James 2:14–26", "focus": "James refuses a version of faith that exists only in words. Living faith becomes visible in obedience. Abraham’s trust moved his hands and feet. Rahab’s faith rearranged her loyalties. Works do not purchase salvation; they reveal that trust has become more than an idea.", "pattern": "", "hidden": "James uses Abraham and Rahab to show that living faith becomes visible. Works do not replace faith; they reveal that faith is alive rather than merely verbal.", "why": "God welcomes dependent prayer as Father and Provider.", "reflect": "Where do I say I trust God but behave as though I do not?", "walk": "Turn one belief into one concrete act of obedience today.", "truth": "James uses Abraham and Rahab to show that living faith becomes visible.", "prayer": "God, make my faith living, active, and visible in the way I walk."}, {"day": 66, "title": "The Mustard Seed and the Kingdom", "reference": "Matthew 13:31–33; 17:20", "focus": "Jesus repeatedly uses small things to describe the surprising work of God. A mustard seed begins almost unnoticed. Leaven works quietly through the dough. Do not despise small obedience because it lacks drama. God’s kingdom often advances through faithful acts that look unimpressive at first.", "pattern": "The seed pattern runs through Scripture: life and fruit emerge from what appears small, buried, or insignificant. Jesus Himself uses seed imagery for the kingdom and for His own death and fruitfulness.", "hidden": "Jesus compares the kingdom to a mustard seed and leaven—small beginnings with disproportionate influence. God often hides scale inside what looks insignificant at first.", "why": "God welcomes dependent prayer as Father and Provider.", "reflect": "What small act of faith have I been dismissing because it seems too little?", "walk": "Do the small faithful thing in front of you today.", "truth": "Jesus compares the kingdom to a mustard seed and leaven—small beginnings with disproportionate influence.", "prayer": "Lord, teach me not to despise beginnings that You can grow."}, {"day": 67, "title": "Faith Through the Storm", "reference": "Mark 4:35–41", "focus": "The disciples had Jesus in the boat and still experienced a storm. His presence did not mean the absence of trouble. Their fear interpreted the storm as evidence that He did not care. Jesus confronted both the wind and their unbelief. Sometimes the deepest question in a storm is not whether God is present, but whether we trust His presence.", "pattern": "", "hidden": "The disciples ask, “Who then is this?” after Jesus stills the storm. The miracle is not merely about weather relief; it presses the identity question—who commands wind and sea?", "why": "God welcomes dependent prayer as Father and Provider.", "reflect": "What storm has made me question God’s care?", "walk": "Pray honestly about the fear, then thank Christ for one evidence of His presence.", "truth": "The disciples ask, “Who then is this?” after Jesus stills the storm.", "prayer": "Jesus, speak peace to what rages around me and faith to what rages within me."}, {"day": 68, "title": "Teach Me to Pray", "reference": "Luke 11:1–13", "focus": "The disciples watched Jesus and asked to be taught to pray. Prayer is not merely instinct; it can be learned. Jesus gives a pattern that begins with the Father’s name and kingdom before moving to daily needs, forgiveness, and protection. Prayer reorders desire before it requests outcomes.", "pattern": "", "hidden": "The disciples do not ask Jesus to teach them to preach or perform miracles; they ask, “Teach us to pray.” They had watched His hidden life with the Father and recognized the source beneath the public ministry.", "why": "God gives wisdom for ordinary life because all of life belongs to Him.", "reflect": "Does my prayer life begin with God’s kingdom or mostly with my emergencies?", "walk": "Pray the Lord’s Prayer slowly, pausing after every phrase to make it personal.", "truth": "The disciples do not ask Jesus to teach them to preach or perform miracles; they ask, “Teach us to pray.” They had watched His hidden life with the Father and recognized the source beneath the public ministry.", "prayer": "Father, teach me to pray until my desires become more aligned with Yours."}, {"day": 69, "title": "Our Father", "reference": "Matthew 6:5–13; Romans 8:14–17", "focus": "Jesus teaches His disciples to begin prayer with relationship: Our Father. Romans says the Spirit bears witness that believers are God’s children. Prayer is not a frightened employee trying to persuade a reluctant boss. It is a son approaching a holy Father with reverence and confidence.", "pattern": "Israel knew God as Father in covenant language, and Christ brings believers into an intimate filial relationship through the Spirit.", "hidden": "Jesus begins prayer with “Our Father.” Romans 8 explains that believers receive the Spirit of adoption. Prayer begins from received relationship, not from trying to persuade a distant deity to become kind.", "why": "God gives wisdom for ordinary life because all of life belongs to Him.", "reflect": "Do I approach God primarily as judge, employer, emergency service, or Father?", "walk": "Spend five minutes addressing God only as Father and thanking Him for access through Christ.", "truth": "Jesus begins prayer with “Our Father.” Romans 8 explains that believers receive the Spirit of adoption.", "prayer": "Father, heal every distorted picture of You that makes me hide instead of come near."}, {"day": 70, "title": "Give Us This Day", "reference": "Matthew 6:11; Exodus 16:13–21", "focus": "Daily bread reaches backward to manna. God trained Israel not to live from hoarded control. Jesus places that same posture inside the disciple’s prayer: give us this day. Dependence is not immaturity when the One depended upon is God.", "pattern": "The prayer for daily bread echoes manna in the wilderness and ultimately points to Christ, the Bread of Life.", "hidden": "The request for daily bread deliberately sounds like manna. Both teach the soul to receive today’s provision without converting tomorrow into today’s anxiety.", "why": "God gives wisdom for ordinary life because all of life belongs to Him.", "reflect": "What am I trying to secure because dependence makes me uncomfortable?", "walk": "Thank God specifically for today’s provision before asking for tomorrow’s.", "truth": "The request for daily bread deliberately sounds like manna.", "prayer": "Give me this day what I need, Lord, and free me from anxious hoarding."}, {"day": 71, "title": "Forgive Us as We Forgive", "reference": "Matthew 6:12, 14–15; Matthew 18:21–35", "focus": "Receiving mercy and extending mercy belong together. Forgiveness does not call evil good, erase healthy boundaries, or guarantee restored trust. It releases the right to personally collect vengeance and places judgment in God’s hands. The forgiven person is called to become a forgiving person.", "pattern": "", "hidden": "Jesus links receiving forgiveness and extending forgiveness because grace is meant to become a new relational economy. The forgiven person is no longer free to live as a professional debt collector of other people’s failures.", "why": "God gives wisdom for ordinary life because all of life belongs to Him.", "reflect": "Whose debt do I keep mentally collecting?", "walk": "Name the offense before God. Ask Him to begin freeing you from the need to punish through resentment.", "truth": "Jesus links receiving forgiveness and extending forgiveness because grace is meant to become a new relational economy.", "prayer": "Father, as You have shown mercy to me, teach me to release vengeance and walk in wisdom."}, {"day": 72, "title": "Pray Without Performing", "reference": "Luke 18:9–14", "focus": "The Pharisee prayed about himself while appearing to address God. The tax collector brought no résumé, only need. Jesus says the humbled tax collector went home justified. Prayer becomes dangerous when it turns into self-congratulation. Grace teaches us to come honestly.", "pattern": "", "hidden": "The tax collector prays almost no words, yet Jesus says he goes home justified. Heaven is not impressed by spiritual performance; humility tells the truth and receives mercy.", "why": "God gives wisdom for ordinary life because all of life belongs to Him.", "reflect": "Do I use spiritual language to avoid honest confession?", "walk": "Pray one short prayer today with no explanation and no self-defense: 'God, be merciful to me.'", "truth": "The tax collector prays almost no words, yet Jesus says he goes home justified.", "prayer": "Lord, strip performance from my prayer life and give me a truthful heart."}, {"day": 73, "title": "The Prayer Jesus Prayed Under Pressure", "reference": "Matthew 26:36–46", "focus": "Gethsemane shows that surrender is not emotional numbness. Jesus names His anguish and asks whether the cup can pass, yet yields: not as I will, but as You will. Mature prayer can be both painfully honest and fully surrendered.", "pattern": "Gethsemane reverses Eden’s rebellion: where humanity grasped for its own will, the obedient Son yields Himself to the Father’s will.", "hidden": "Gethsemane is a garden where the obedient Son says yes to the Father’s will. Christian readers have long noticed the contrast with Eden: humanity grasped its own will there; Christ yields His will here on the way to redeem humanity.", "why": "God gives wisdom for ordinary life because all of life belongs to Him.", "reflect": "What request do I need to pray honestly and then surrender?", "walk": "Pray it in two sentences: 'Father, this is what I desire. Yet I entrust myself to Your will.'", "truth": "Gethsemane is a garden where the obedient Son says yes to the Father’s will.", "prayer": "Jesus, teach me the courage of surrendered prayer."}, {"day": 74, "title": "Intercede for Your House", "reference": "Job 1:1–5; John 17:6–19", "focus": "Job regularly brought his children before God. Jesus prayed specifically for those entrusted to Him. You cannot force another person’s spiritual response, but you can faithfully carry your family before the Lord. Intercession changes the posture of leadership from control to dependence.", "pattern": "", "hidden": "Job regularly brings his children before God, and Jesus’ longest recorded prayer includes detailed intercession for His disciples. Spiritual leadership carries names into prayer instead of trying to control outcomes.", "why": "God gives wisdom for ordinary life because all of life belongs to Him.", "reflect": "Do I lecture my family more than I pray for them?", "walk": "Pray by name for each person in your household, asking for God’s work in their heart and for your own faithfulness toward them.", "truth": "Job regularly brings his children before God, and Jesus’ longest recorded prayer includes detailed intercession for His disciples.", "prayer": "Lord, watch over my household and make me a faithful intercessor, not a controlling one."}, {"day": 75, "title": "Wisdom Begins With the Fear of the Lord", "reference": "Proverbs 1:1–7; 9:10", "focus": "Biblical wisdom is not merely cleverness, age, or accumulated experience. Its beginning is the fear of the Lord: reverent recognition that God is God and we are not. Wisdom grows when His character and commands become the fixed point by which choices are measured.", "pattern": "", "hidden": "Proverbs says fear of the LORD is the beginning of wisdom, not the advanced course. Wisdom starts by putting God in the center before it evaluates the problem.", "why": "God is a reconciler who forms people of truth, forgiveness, honor, and peace.", "reflect": "Where have experience or opinion become more authoritative to me than Scripture?", "walk": "Before one important decision today, ask: 'What honors the Lord here?'", "truth": "Proverbs says fear of the LORD is the beginning of wisdom, not the advanced course.", "prayer": "God, make reverence for You the beginning of every wisdom I claim to possess."}, {"day": 76, "title": "Faithful Work", "reference": "Genesis 2:15; Colossians 3:22–24", "focus": "Work appears before the fall. Labor is not itself a curse; frustrated toil is. Scripture dignifies ordinary work when it is done unto the Lord. A person can worship God through reliability, craftsmanship, honesty, patience, and service even when the task seems small.", "pattern": "Adam is placed in the garden to work and keep it. Christ, the last Adam, restores human vocation under God rather than abolishing meaningful labor.", "hidden": "Work appears in Genesis before the fall. The curse affects toil, but vocation itself is older than sin; meaningful work belongs to God’s good creation.", "why": "God is a reconciler who forms people of truth, forgiveness, honor, and peace.", "reflect": "What would change if I treated today’s work as service to Christ?", "walk": "Do one ordinary task exceptionally well without needing recognition.", "truth": "Work appears in Genesis before the fall.", "prayer": "Lord, receive the work of my hands as an offering of faithfulness."}, {"day": 77, "title": "Steward, Not Owner", "reference": "Psalm 24:1; Matthew 25:14–30", "focus": "The earth and everything in it belong to the Lord. Jesus’ parable of the talents portrays servants entrusted with another’s property. Time, money, abilities, influence, relationships, and opportunities are stewardship before they are possession. That changes both pride and fear.", "pattern": "", "hidden": "Psalm 24 says the earth is the LORD’s; Jesus’ stewardship parables describe servants managing another’s property. Ownership language changes when everything is first received as trust.", "why": "God is a reconciler who forms people of truth, forgiveness, honor, and peace.", "reflect": "What have I been treating as mine rather than entrusted to me?", "walk": "Choose one resource—time, money, skill, or influence—and use a portion of it deliberately for someone else’s good.", "truth": "Psalm 24 says the earth is the LORD’s; Jesus’ stewardship parables describe servants managing another’s property.", "prayer": "Owner of all things, make me faithful with what passes through my hands."}, {"day": 78, "title": "Money Reveals the Heart", "reference": "Matthew 6:19–24; 1 Timothy 6:6–10, 17–19", "focus": "Jesus connects treasure with the heart because money exposes what we trust, fear, love, and pursue. Scripture does not call money evil; it warns against loving it and trusting riches. Generosity loosens the grip of possessions by reminding the soul that God is Provider.", "pattern": "", "hidden": "Jesus places treasure and heart in the same sentence. Money becomes diagnostic: where treasure goes, attention, imagination, fear, and hope tend to follow.", "why": "God is a reconciler who forms people of truth, forgiveness, honor, and peace.", "reflect": "What does my spending reveal about what I value most?", "walk": "Give something intentionally today—not impulsively, but worshipfully.", "truth": "Jesus places treasure and heart in the same sentence.", "prayer": "God, keep money in my hand and out of my throne."}, {"day": 79, "title": "The Wisdom of Restraint", "reference": "Proverbs 10:19; 17:27–28; James 3:1–12", "focus": "A mature believer does not say everything he could say. Proverbs connects many words with increased opportunity for sin. James compares the tongue to a rudder and fire. Restraint is not weakness; it is strength that has learned it does not need to discharge every thought.", "pattern": "", "hidden": "James compares the tongue to a small rudder steering a large ship. Scripture repeatedly treats small speech habits as capable of redirecting entire relationships.", "why": "God is a reconciler who forms people of truth, forgiveness, honor, and peace.", "reflect": "Where does my mouth regularly outrun wisdom?", "walk": "Leave one unnecessary criticism unsaid today.", "truth": "James compares the tongue to a small rudder steering a large ship.", "prayer": "Lord, make my words few enough to be trustworthy and gracious enough to heal."}, {"day": 80, "title": "Counsel and Teachable Strength", "reference": "Proverbs 11:14; 15:22; 19:20", "focus": "Scripture repeatedly associates safety and success with wise counsel. Independence can masquerade as strength while actually protecting blind spots. Teachable people can receive input without surrendering discernment. Pride needs to be the smartest person in the room; wisdom knows it has something to learn.", "pattern": "", "hidden": "Proverbs says plans are established in the presence of many counselors. Teachability is not indecision; it is strength humble enough to admit one pair of eyes has blind spots.", "why": "God is a reconciler who forms people of truth, forgiveness, honor, and peace.", "reflect": "Whose wise counsel have I resisted because it challenged me?", "walk": "Ask one mature believer for honest input on a decision or pattern.", "truth": "Proverbs says plans are established in the presence of many counselors.", "prayer": "God, make me strong enough to remain teachable."}, {"day": 81, "title": "Rest Is Also Obedience", "reference": "Genesis 2:1–3; Mark 2:23–28; Hebrews 4:9–11", "focus": "God built rest into creation before human exhaustion existed. Sabbath teaches that the world continues without our constant effort. Jesus says Sabbath was made for man, not man for Sabbath. Hebrews deepens the image into the rest found in God’s completed work.", "pattern": "Creation rest, Sabbath, Canaan, and Hebrews’ promised rest form a biblical pattern that culminates in trusting God’s finished provision.", "hidden": "Genesis ends creation with rest, and Hebrews later speaks of entering God’s rest. Rest is woven into creation and redemption as trust in work God has completed, not merely recovery from work we completed.", "why": "God is a reconciler who forms people of truth, forgiveness, honor, and peace.", "reflect": "What does my inability to stop reveal about what I believe depends on me?", "walk": "Create one hour today with no work, no news, and no problem-solving. Receive it as a gift.", "truth": "Genesis ends creation with rest, and Hebrews later speaks of entering God’s rest.", "prayer": "Lord of the Sabbath, teach me to work faithfully and rest trustfully."}, {"day": 82, "title": "The Ministry of Reconciliation", "reference": "2 Corinthians 5:17–21", "focus": "God reconciled us to Himself through Christ and then entrusted believers with a ministry of reconciliation. Reconciliation is larger than avoiding arguments. It means becoming a person through whom truth and grace move toward repaired relationship whenever possible.", "pattern": "The gospel itself is reconciliation: enemies brought near through Christ. Human reconciliation takes its shape from what God has done first.", "hidden": "2 Corinthians says God reconciled us to Himself and then gave us the ministry of reconciliation. We do not invent reconciliation; we become carriers of what God has already done in Christ.", "why": "God keeps promises across generations.", "reflect": "Where can I become an agent of peace rather than an amplifier of distance?", "walk": "Take one low-pressure step toward peace: a kind message, an apology, a calm invitation, or a prayer.", "truth": "2 Corinthians says God reconciled us to Himself and then gave us the ministry of reconciliation.", "prayer": "God of reconciliation, make me a person through whom peace can travel."}, {"day": 83, "title": "Own Your Part", "reference": "Matthew 7:1–5; Psalm 139:23–24", "focus": "Jesus’ image of the plank and speck does not forbid discernment; it forbids self-blindness. Restoration accelerates when a person becomes more interested in their own obedience than in winning the case against another person. Psalm 139 gives us the brave prayer: search me.", "pattern": "", "hidden": "Jesus’ plank-and-speck image is intentionally disproportionate. Before correcting another person, discipleship begins by allowing God to perform eye surgery on the one doing the correcting.", "why": "God keeps promises across generations.", "reflect": "What part of this conflict belongs to me even if the other person also has a part?", "walk": "Write your part in one paragraph without using the words 'but they.'", "truth": "Jesus’ plank-and-speck image is intentionally disproportionate.", "prayer": "Search me, O God. Show me what is mine to confess, repair, and change."}, {"day": 84, "title": "Speak the Truth in Love", "reference": "Ephesians 4:11–16, 25–32", "focus": "Biblical love and biblical truth are not enemies. Truth without love can become brutality; love without truth can become avoidance. Paul holds them together because growth into Christ requires both. The goal of hard conversation is not release of anger but the good of the relationship and the honor of God.", "pattern": "", "hidden": "Ephesians says to speak truth “in love” because Christian maturity requires both. Truth severed from love and love severed from truth are both distortions of Christ.", "why": "God keeps promises across generations.", "reflect": "Do I usually sacrifice truth for peace, or love for the satisfaction of saying what I think?", "walk": "Before a difficult conversation, write the truthful sentence and then rewrite it with gentleness.", "truth": "Ephesians says to speak truth “in love” because Christian maturity requires both.", "prayer": "Jesus, make my truth loving and my love truthful."}, {"day": 85, "title": "Bless Instead of Curse", "reference": "Romans 12:14–21; 1 Peter 3:8–12", "focus": "The New Testament repeatedly calls believers to answer hostility with blessing rather than retaliation. This does not remove boundaries or justice. It removes personal vengeance as the governing spirit. Refusing to curse someone protects your own heart from becoming shaped by their wrong.", "pattern": "", "hidden": "Romans 12 does not define overcoming evil as defeating the evildoer; it says overcome evil with good. The victory is refusing to let another person’s sin reproduce itself in you.", "why": "God keeps promises across generations.", "reflect": "Whom have I been punishing in my imagination?", "walk": "Pray one sincere blessing over that person’s life without denying the wrong.", "truth": "Romans 12 does not define overcoming evil as defeating the evildoer; it says overcome evil with good.", "prayer": "Lord, free me from becoming like what hurt me."}, {"day": 86, "title": "Honor in the Home", "reference": "1 Peter 3:7; Romans 12:10", "focus": "Honor means recognizing God-given worth and treating another person accordingly. In marriage, contempt is corrosive because it reduces a person to their failures. Peter treats conduct inside marriage as spiritually serious, even tying a husband’s treatment of his wife to his prayers. Honor does not require pretending sin is acceptable; it requires refusing dehumanization.", "pattern": "", "hidden": "Peter tells husbands that how they honor their wives is connected to their prayers. Scripture refuses to separate spiritual life from treatment of the person at home.", "why": "God keeps promises across generations.", "reflect": "Do the people closest to me experience respect in my words, face, tone, and behavior?", "walk": "Express one specific, truthful honor today without attaching a request to it.", "truth": "Peter tells husbands that how they honor their wives is connected to their prayers.", "prayer": "God, teach me to see and treat the people closest to me with the dignity You require."}, {"day": 87, "title": "Peace Requires Courage", "reference": "Matthew 5:9, 23–24; Romans 14:19", "focus": "Peacemaking is not passive. Jesus blesses peacemakers, not merely peace-lovers. Sometimes peace requires initiating repair, confessing first, setting a boundary, refusing gossip, or entering a conversation you would rather avoid. Biblical peace is built with courage and truth.", "pattern": "", "hidden": "Jesus blesses peacemakers, not merely people who prefer quiet. Biblical peace sometimes requires courageous movement toward truth, confession, repair, or boundary.", "why": "God keeps promises across generations.", "reflect": "What brave action might peace require from me?", "walk": "Take one peaceful action that costs you pride.", "truth": "Jesus blesses peacemakers, not merely people who prefer quiet.", "prayer": "Prince of Peace, make me courageous enough to build what I say I want."}, {"day": 88, "title": "A Household of Grace", "reference": "Colossians 3:12–21", "focus": "Paul brings compassion, kindness, humility, patience, forgiveness, love, peace, gratitude, and the Word of Christ into the household. A Christian home is not a place where nobody fails; it is a place where grace governs what people do after failure. Leadership is measured partly by the culture a person helps create.", "pattern": "", "hidden": "Colossians places the Word of Christ in the middle of household life. A grace-shaped home is not one without failure; it is one where compassion, forgiveness, gratitude, and truth govern what happens after failure.", "why": "God keeps promises across generations.", "reflect": "What atmosphere do I contribute when something goes wrong at home?", "walk": "Choose one word from Colossians 3—kindness, patience, forgiveness, gratitude—and make it the tone of your home today.", "truth": "Colossians places the Word of Christ in the middle of household life.", "prayer": "Christ, let Your Word dwell richly in me until my home feels its effects."}, {"day": 89, "title": "The Promise of the Coming Seed", "reference": "Genesis 3:14–15; Galatians 4:4–5", "focus": "The first promise of conflict and eventual victory appears immediately after the fall. Scripture then unfolds a long expectation: a coming offspring, a promised Son, a King, a Servant. Christmas does not begin in Bethlehem. It belongs to a promise God was unfolding from the beginning.", "pattern": "Genesis 3:15 is often called the first gospel promise because later Scripture reveals the victory God brings through the coming Christ.", "hidden": "Genesis 3 introduces the promised offspring in the very chapter where sin enters. The Bible’s rescue story begins before humanity leaves Eden: judgment and promise stand in the same scene.", "why": "In Jesus, God has come near as King, Bread, Light, and Savior.", "reflect": "What does it say about God that He spoke promise into the very scene of human failure?", "walk": "Trace one promise of Christ through Genesis 3:15, Isaiah 7:14, and Galatians 4:4–5.", "truth": "Genesis 3 introduces the promised offspring in the very chapter where sin enters.", "prayer": "Promise-keeping God, help me trust what You say even when fulfillment takes time."}, {"day": 90, "title": "A Son Is Given", "reference": "Isaiah 7:14; 9:1–7; Matthew 1:18–25", "focus": "Isaiah speaks into political fear and national darkness with the promise of a child whose identity and reign exceed ordinary kingship. Matthew sees Jesus’ birth within that prophetic horizon. God’s answer to darkness is not only a change of circumstance; it is the arrival of His Son.", "pattern": "The promised child and Davidic King in Isaiah prepare the categories Matthew uses to announce Jesus as Immanuel—God with us.", "hidden": "Isaiah’s promised child carries titles too large for an ordinary king. Matthew’s birth narrative invites readers to see Jesus inside that long prophetic hope.", "why": "In Jesus, God has come near as King, Bread, Light, and Savior.", "reflect": "Where do I need more than improved circumstances—I need the presence of Christ?", "walk": "Read Isaiah 9:6 slowly and pray through each title given to the promised Son.", "truth": "Isaiah’s promised child carries titles too large for an ordinary king.", "prayer": "Immanuel, be God with me in every place I have called dark."}, {"day": 91, "title": "Bethlehem: God Works Through the Small", "reference": "Micah 5:2–5; Matthew 2:1–6", "focus": "Micah names little Bethlehem as the place from which the ruler would come. God repeatedly overturns human assumptions about significance. Bethlehem is small, David was the youngest, and a manger does not look like a throne. God’s glory is not dependent on human impressiveness.", "pattern": "Bethlehem binds David’s story to the greater Son of David. The shepherd king’s town becomes the birthplace of the final Shepherd-King.", "hidden": "Bethlehem means “house of bread.” The One born there later says, “I am the bread of life.” Scripture does not explicitly call the town-name a prophecy, but the linguistic fact beautifully echoes the Bible’s Bread-of-Life theme.", "why": "In Jesus, God has come near as King, Bread, Light, and Savior.", "reflect": "What small or unimpressive place in my life have I assumed God cannot use?", "walk": "Thank God for one ordinary place, person, or responsibility through which He may be working.", "truth": "Bethlehem means “house of bread.” The One born there later says, “I am the bread of life.” Scripture does not explicitly call the town-name a prophecy, but the linguistic fact beautifully echoes the Bible’s Bread-of-Life theme.", "prayer": "Lord, keep me from overlooking Your work because it arrives without spectacle."}, {"day": 92, "title": "The Word Became Flesh", "reference": "John 1:1–18", "focus": "John takes us beyond the manger to the mystery beneath it: the eternal Word became flesh and dwelt among us. The Creator entered creation. The One through whom all things were made came near enough to be seen, heard, touched, rejected, and received. Christianity rests not on humanity climbing to God, but on God coming to us.", "pattern": "John’s language of 'dwelling' echoes the tabernacle. The God whose glory filled the sanctuary now makes His dwelling among us in the Son.", "hidden": "John says the Word “dwelt” among us; the Greek verb carries the idea of pitching a tent or tabernacling. The presence once associated with the tabernacle is now encountered in the person of Jesus.", "why": "In Jesus, God has come near as King, Bread, Light, and Savior.", "reflect": "What does the incarnation teach me about God’s willingness to come near?", "walk": "Read John 1:1–18 aloud and underline every statement about who Jesus is.", "truth": "John says the Word “dwelt” among us; the Greek verb carries the idea of pitching a tent or tabernacling.", "prayer": "Jesus, let the wonder of Your coming become more than a familiar story to me."}, {"day": 93, "title": "Christmas Eve: The Long Night Before the Announcement", "reference": "Luke 1:67–79; Isaiah 60:1–3", "focus": "Zechariah describes humanity as sitting in darkness and in the shadow of death, awaiting a sunrise from on high. Scripture does not sentimentalize darkness; it announces light into it. The coming of Christ means darkness is real but not final.", "pattern": "Dawn and light imagery runs from the prophets into the Gospel announcement: God visits His people like sunrise breaking into night.", "hidden": "Zechariah describes the Messiah’s coming as a sunrise visiting people in darkness. Christmas is presented not as seasonal sentiment but as dawn breaking into the shadow of death.", "why": "In Jesus, God has come near as King, Bread, Light, and Savior.", "reflect": "What darkness am I tempted to call final?", "walk": "Before bed, sit in a dim room and read Luke 1:78–79. Thank God for the Light who came.", "truth": "Zechariah describes the Messiah’s coming as a sunrise visiting people in darkness.", "prayer": "God of dawn, shine into every place where I have surrendered to despair."}, {"day": 94, "title": "Christmas Day: Unto You Is Born a Savior", "reference": "Luke 2:1–20", "focus": "The angel’s announcement is intensely personal: unto you is born a Savior. Heaven’s glory appears not in Caesar’s palace but among shepherds, surrounding the birth of a child laid in a feeding trough. The sign of God’s kingdom is humility wrapped around glory.", "pattern": "Luke contrasts earthly empire and the true King. The titles Savior and Lord, claimed by rulers, are announced over the child in Bethlehem.", "hidden": "Jesus is laid in a manger—a feeding trough—in Bethlehem, the “house of bread.” Luke does not explain this as a type, so we should not force it, but beside John 6 it forms a striking Christ-centered meditation on the Bread who came near.", "why": "In Jesus, God has come near as King, Bread, Light, and Savior.", "reflect": "Have I become so familiar with Christmas that I no longer marvel at the humility of God?", "walk": "Read Luke 2 slowly with your family if possible. Thank Jesus aloud for coming near.", "truth": "Jesus is laid in a manger—a feeding trough—in Bethlehem, the “house of bread.” Luke does not explain this as a type, so we should not force it, but beside John 6 it forms a striking Christ-centered meditation on the Bread who came near.", "prayer": "Savior and Lord, receive my worship. Make Your humility the pattern of my strength."}, {"day": 95, "title": "Behold the Lamb of God", "reference": "John 1:29–36; Exodus 12:1–13; Revelation 5:6–14", "focus": "The Bible’s Lamb imagery stretches from Passover to John the Baptist to the throne room of Revelation. The Lamb is sacrificed and victorious. The same Jesus whose birth we celebrate came with a redemptive mission. Christmas and the cross belong to one story.", "pattern": "Passover lamb → Lamb of God → slain and reigning Lamb is one of Scripture’s clearest beginning-to-end redemptive patterns.", "hidden": "The Lamb theme runs from Passover to John the Baptist to Revelation’s throne room. The Lamb is both slain and reigning; sacrifice and victory are not opposites in the gospel.", "why": "God finishes what He begins and leads His people toward new creation.", "reflect": "Do I receive Jesus mainly as an inspiration, or as the Lamb who takes away sin?", "walk": "Read the three passages in sequence and write what each adds to your understanding of Jesus.", "truth": "The Lamb theme runs from Passover to John the Baptist to Revelation’s throne room.", "prayer": "Lamb of God, thank You for coming not merely to teach me, but to redeem me."}, {"day": 96, "title": "The New Covenant", "reference": "Jeremiah 31:31–34; Luke 22:14–20; Hebrews 8:6–13", "focus": "Jeremiah promises a covenant in which God’s law is written on hearts and sins are remembered no more. At the Last Supper, Jesus identifies the cup with the new covenant in His blood. Hebrews explicitly joins these passages. The covenant rests on God’s action before it produces ours.", "pattern": "Jeremiah’s promise → Jesus’ covenant blood → Hebrews’ explanation forms an explicit biblical fulfillment chain.", "hidden": "Jeremiah promises a new covenant; Jesus names the cup “the new covenant in My blood”; Hebrews quotes Jeremiah at length. This is not a speculative connection—the New Testament explicitly interprets the promise through Christ.", "why": "God finishes what He begins and leads His people toward new creation.", "reflect": "Do I live as though forgiveness depends on my ability to keep paying for what Christ has covered?", "walk": "Read Hebrews 8:12 twice and thank God specifically for the promise of forgiven sin.", "truth": "Jeremiah promises a new covenant; Jesus names the cup “the new covenant in My blood”; Hebrews quotes Jeremiah at length.", "prayer": "Covenant-keeping God, write Your ways on my heart and teach me to live as a forgiven person."}, {"day": 97, "title": "He Makes All Things New", "reference": "Isaiah 43:18–19; 2 Corinthians 5:17; Revelation 21:1–5", "focus": "Biblical newness is not denial of history. God does not ask Israel to pretend the former things never happened; He announces that His creative power is not exhausted by them. In Christ there is new creation, and Revelation points toward the renewal of all things.", "pattern": "Creation, new creation in Christ, and final new creation belong to one redemptive arc: God restores what sin has damaged and brings His purposes to completion.", "hidden": "The Bible opens with creation and closes with new creation. Revelation does not abandon Genesis; it resolves its losses—God dwelling with humanity, the curse removed, and the tree of life restored.", "why": "God finishes what He begins and leads His people toward new creation.", "reflect": "What old chapter am I still treating as the limit of what God can do next?", "walk": "Write one sentence beginning, 'Because God makes things new, I will not assume…'", "truth": "The Bible opens with creation and closes with new creation.", "prayer": "Creator and Redeemer, keep my past from becoming an idol of limitation."}, {"day": 98, "title": "Number Your Days", "reference": "Psalm 90:1–17", "focus": "Moses contrasts God’s everlasting nature with the brevity of human life. 'Teach us to number our days' is not morbid; it is a request for wisdom. Limited days become meaningful when they are consciously offered to God. The answer to mortality is not panic but faithful stewardship.", "pattern": "", "hidden": "Psalm 90 is attributed to Moses, a man who watched an entire generation pass through the wilderness. “Teach us to number our days” comes from someone who understood both mortality and the faithfulness of the everlasting God.", "why": "God finishes what He begins and leads His people toward new creation.", "reflect": "If I truly believed my days were a gift, what would I stop postponing?", "walk": "Write three priorities for the coming year that reflect eternity rather than mere urgency.", "truth": "Psalm 90 is attributed to Moses, a man who watched an entire generation pass through the wilderness.", "prayer": "Lord, teach me to number my days so that wisdom shapes how I spend them."}, {"day": 99, "title": "Lay Down the Old Year", "reference": "Philippians 3:7–14; Isaiah 43:18–19", "focus": "Paul refuses to live either from past achievements or past failures. He counts all things in relation to Christ and presses forward. You do not honor the past by remaining trapped in it. There is a holy way to remember and a holy way to release.", "pattern": "", "hidden": "Paul can remember the past without being ruled by it. Philippians 3 holds two movements together: knowing what came before and refusing to let it become the horizon of what God can still do.", "why": "God finishes what He begins and leads His people toward new creation.", "reflect": "What do I need to carry as wisdom but stop carrying as identity?", "walk": "Divide a page into two columns: 'Take Forward' and 'Leave Behind.' Fill both honestly.", "truth": "Paul can remember the past without being ruled by it.", "prayer": "Jesus, help me keep the lessons without carrying chains You never asked me to keep."}, {"day": 100, "title": "Day 100 — Finish the Year Looking Unto Jesus", "reference": "Hebrews 12:1–3; Philippians 1:3–6; Revelation 22:12–21", "focus": "One hundred days end where Christian life must always return: Jesus. Hebrews says to run with endurance while looking to Him. Paul trusts that God will complete the good work He began. Revelation closes Scripture with Christ coming, grace offered, and the church answering, 'Come.' A calendar ends; discipleship does not. Tomorrow is not Day 101 of a program so much as another day of abiding in Christ.", "pattern": "Scripture ends by pointing forward. Eden’s lost access, prophetic promise, the Lamb’s redemption, and the new creation converge in the final hope: Christ with His people forever.", "hidden": "Revelation closes not with humanity escaping God’s purposes but with Christ coming, grace extended, and creation restored. The final posture of Scripture is forward-looking hope: “Come, Lord Jesus.”", "why": "God finishes what He begins and leads His people toward new creation.", "reflect": "After one hundred days, what has God most clearly been asking me to become?", "walk": "Write a final prayer of surrender for the coming year. Choose the next Bible book you will begin tomorrow and the daily time you will protect.", "truth": "Revelation closes not with humanity escaping God’s purposes but with Christ coming, grace extended, and creation restored.", "prayer": "Lord Jesus, thank You for carrying me to the end of this year. The coming year belongs to You. Keep my eyes on You, my heart teachable, my marriage and family under Your grace, my mind anchored in Your Word, and my life faithful until I see You face to face. Amen."}]
print(f'Loaded {len(NARRATION_DATA)} devotional narration scripts.')


In [ ]:
import re, os, math, subprocess, json
import torch
import torchaudio
from IPython.display import Audio, display

def spoken_reference(ref):
    """Turn common Bible reference notation into more natural spoken English."""
    parts = [p.strip() for p in ref.replace("–","-").split(";") if p.strip()]
    spoken = []
    for part in parts:
        m = re.match(r"^(.+?)\s+(\d+)(?::(\d+)(?:-(\d+))?)?(?:-(\d+))?$", part)
        if not m:
            spoken.append(part.replace(":", ", "))
            continue
        book, ch, v1, v2, ch2 = m.groups()
        if ch2 and not v1:
            spoken.append(f"{book}, chapters {ch} through {ch2}")
        elif v1 and v2:
            spoken.append(f"{book}, chapter {ch}, verses {v1} through {v2}")
        elif v1:
            spoken.append(f"{book}, chapter {ch}, verse {v1}")
        else:
            spoken.append(f"{book}, chapter {ch}")
    return "; and ".join(spoken)

def day_script(d):
    sections = [
        f"Hey, it's Libby. Day {d['day']}. {d['title']}.",
        f"Today's Word is {spoken_reference(d['reference'])}.",
        "Understanding the Word. " + d["focus"],
        "Hidden in plain sight. " + d["hidden"],
    ]
    if d.get("pattern"):
        sections.append("Type, shadow, or biblical pattern. " + d["pattern"])
    sections += [
        "Why it matters. " + d["why"],
        "Today's focus. Reflect. " + d["reflect"],
        "Walk it out. " + d["walk"],
        "Truth to carry today. " + d["truth"],
        "Let's pray. " + d["prayer"],
    ]
    return sections

def split_for_voice(text, max_chars=330):
    """Split on sentence boundaries so Chatterbox stays natural and stable on long devotionals."""
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= max_chars:
        return [text]
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, buf = [], ""
    for sentence in sentences:
        if not sentence:
            continue
        candidate = (buf + " " + sentence).strip()
        if len(candidate) <= max_chars:
            buf = candidate
        else:
            if buf:
                chunks.append(buf)
            if len(sentence) <= max_chars:
                buf = sentence
            else:
                words = sentence.split()
                piece = ""
                for w in words:
                    cand = (piece + " " + w).strip()
                    if len(cand) <= max_chars:
                        piece = cand
                    else:
                        if piece:
                            chunks.append(piece)
                        piece = w
                buf = piece
    if buf:
        chunks.append(buf)
    return chunks

def generate_piece(text):
    wav = model.generate(
        text,
        audio_prompt_path="/content/libby-reference.wav",
        exaggeration=EXAGGERATION,
        cfg_weight=CFG_WEIGHT,
        temperature=TEMPERATURE,
    )
    return wav.cpu()

def silence(ms):
    frames = int(model.sr * (ms / 1000.0))
    return torch.zeros((1, frames), dtype=torch.float32)

def build_day_audio(d):
    day_num = d["day"]
    basename = f"day-{day_num:03d}-libby"
    wav_path = os.path.join(OUTPUT_FOLDER, basename + ".wav")
    mp3_path = os.path.join(OUTPUT_FOLDER, basename + ".mp3")
    txt_path = os.path.join(OUTPUT_FOLDER, basename + ".txt")

    if os.path.exists(mp3_path) and os.path.getsize(mp3_path) > 10000:
        print(f"Day {day_num}: already finished — skipping.")
        return mp3_path

    sections = day_script(d)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(sections))

    audio_parts = []
    for section_index, section in enumerate(sections):
        chunks = split_for_voice(section)
        for chunk_index, chunk in enumerate(chunks):
            print(f"  Day {day_num:03d} · section {section_index+1}/{len(sections)} · chunk {chunk_index+1}/{len(chunks)}")
            audio_parts.append(generate_piece(chunk))
            if chunk_index < len(chunks)-1:
                audio_parts.append(silence(PAUSE_BETWEEN_CHUNKS_MS))
        if section_index < len(sections)-1:
            audio_parts.append(silence(PAUSE_BETWEEN_SECTIONS_MS))

    full = torch.cat(audio_parts, dim=1)
    torchaudio.save(wav_path, full, model.sr)

    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", wav_path,
        "-codec:a", "libmp3lame",
        "-b:a", "128k",
        mp3_path
    ], check=True)

    print(f"Day {day_num}: saved.")
    return mp3_path

print("Narration helpers ready.")


In [ ]:
print("5/5 — Generating Types & Shadows in Libby's cloned voice.")
print("Each completed day is saved immediately in the Colab session.")
print("Keep the Colab session open while generation runs.\n")

completed_paths = []

for d in NARRATION_DATA:
    n = d["day"]
    if n < START_DAY or n > END_DAY:
        continue
    print("\n" + "="*60)
    print(f"DAY {n}: {d['title']}")
    print("="*60)
    path = build_day_audio(d)
    completed_paths.append(path)

print("\nAll selected days are finished.")
print(f"Audio folder: {OUTPUT_FOLDER}")

# Write a simple manifest the website can use later.
manifest = {
    str(d["day"]): {
        "title": d["title"],
        "audio": f"day-{d['day']:03d}-libby.mp3"
    }
    for d in NARRATION_DATA
    if START_DAY <= d["day"] <= END_DAY
}
manifest_path = os.path.join(OUTPUT_FOLDER, "audio-manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Manifest saved:", manifest_path)

# Play the first selected finished day.
first_path = next((p for p in completed_paths if os.path.exists(p)), None)
if first_path:
    print("\nPreview of the first generated day:")
    display(Audio(first_path))


In [ ]:
# OPTIONAL: create one ZIP containing the website-ready MP3 files and manifest.
import os, zipfile

zip_path = os.path.join(OUTPUT_FOLDER, "Types-and-Shadows-Libby-Audio.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(OUTPUT_FOLDER)):
        if name.endswith(".mp3") or name == "audio-manifest.json":
            z.write(os.path.join(OUTPUT_FOLDER, name), arcname=name)

print("Website-ready ZIP created:")
print(zip_path)
